In [ ]:
# citation_format_experiment.ipynb
#
# [실험 목적]
# 1) 팀 채점기가 Citation Coverage를 0%로 매기는 이유가 정말 "인용을
#    안 해서"인지, 아니면 형식만 안 맞는 건지 확인
# 2) "학교에서 발주한 사업 알려줘"처럼 조건에 맞는 문서를 전부 찾아야
#    하는 질문(목록형·집계형)이 지금 파이프라인으로 왜 정답을 다 못
#    찾는지, 그리고 랭킹/기간 조건이 섞인 질문까지 처리할 수 있는지 확인
#
# [진행 방식과 알아낸 것]
#
# 1. Citation Coverage 0% 원인 조사 (cell 9~19)
#    - 기존 답변들을 직접 훑어보니 실제로는 정답 문서명을 78.63% 비율로
#      언급하고 있었음 -> "인용을 안 한다"가 아니라 형식이 "근거:",
#      "근거 문서:", "출처:" 등으로 제각각인 게 원인이라고 판단
#    - 프롬프트 지시를 "[근거: 문서명1, 문서명2]" 정확한 형식으로 못박고
#      다른 표현은 쓰지 말라고 명시
#    - 추가한 것: g25(랭킹형 질문의 자동 계산 반환 문자열)도 같은 형식으로 통일
#    - 알아낸 것: core40/rag-56 재검증 결과 정답 점수는 그대로 유지되고
#      형식만 정확히 통일됨 -> 형식 문제였다는 게 확인됨
#
#
# 2. 목록형·집계형 질문(set-13) 테스트 및 개선 (cell 24~48)
#    - "학교 발주 사업 알려줘" 등 6개 질문을 우선 기존 파이프라인(벡터
#      검색)으로 테스트 -> 정답 12개 문서 중 5개만 나오는 문제를 발견
#    - is_school_org() 함수를 새로 만듦. "대학"이라는 단어만 보면
#      "대학스포츠협의회" 같은 협회까지 오탐되길래, "협의회"/"협회"가
#      들어간 기관명은 제외하도록 조건을 추가
#    - k(검색 개수)를 80->200으로 늘려봐도 대전대(MILE 플랫폼) 문서가
#      계속 안 나오는 걸 확인 -> k를 늘리는 걸로는 해결이 안 되고,
#      이 문서 자체가 질문과 벡터 유사도가 낮아서 순위 안에 아예 안
#      든다는 걸 직접 검색해서 확인함
#    - 그래서 조건 필터형 질문(conditions 분기) 처리 방식을 벡터 검색에서
#      "메타데이터에서 조건에 맞는 문서를 직접 다 찾아내는" 방식으로
#      완전히 바꿈 -> b12(학교) 12개 문서 전부 정확히 찾아냄
#    - 여기서 추가로 발견한 것: 긴급/보안/재난 조건은 extract_filter_
#      conditions에서 감지는 되고 있었는데, 실제로 필터링하는 build_meta_
#      filter 함수엔 이 조건들을 검사하는 코드가 아예 없어서 필터가
#      사실상 작동을 안 하고 있었음. 파일명에 "긴급"/"보안"/"재난"이
#      있는지 직접 확인하는 코드를 추가해서 해결
#    - 알아낸 것: b14(긴급)/b16(보안)/b21(재난+금액)/b24(지자체+금액)를
#      재검증한 결과 전부 정답과 정확히 일치함

#
# 3. "N일/N개월 이내" 기간 조건 질문 처리 신규 구현 (cell 49~63)
#    - 메타데이터에는 사업기간 정보가 없어서, 문서 본문에서 "계약일/
#      착수일로부터 N일" 같은 문장을 정규식으로 직접 찾아 파싱하는
#      extract_period_days() 함수를 새로 만듦
#    - 처음 만든 정규식은 "계약체결일"이라는 표현을 못 잡았고, "~"
#      (물결표) 구분자도 놓쳤고, "계약일로부터 00일"처럼 서식에 값이
#      안 채워진 플레이스홀더를 0일로 잘못 읽는 문제가 순서대로
#      발견돼서 그때그때 정규식을 v1->v2->v3로 보정함
#    - b15("3개월 이내 짧은 사업")를 검증하다가, 우리가 답변에서 초과로
#      찾은 3개 문서(전북대 75일·정읍시 80일·파주 60일)를 원문까지 직접
#      대조해보니 실제로 다 조건을 만족하는 정답이었음 -> 이건
#      로직 문제가 아니라 골든셋 정답 목록 자체가 3건 누락돼 있다는
#      뜻이라 팀에 공유가 필요한 사항으로 정리함
#
#
# 4. Visual(표/그림) 문항 텍스트 파이프라인 한계 확인 (cell 66~67)
#    - 표(table) 유형 4건은 텍스트 파싱만으로도 부분적인 답변이 나옴
#    - 그림(figure) 유형 4건은 전부 "확인되지 않습니다"로 기권함
#    - 알아낸 것: 그림은 이미지 자체가 텍스트로 파싱이 안 되니까
#      generation 파이프라인으로는 원천적으로 처리가 불가능하고, VLM/OCR
#      통합이 있어야 풀린다는 걸 확인 -> Generation 영역 밖의
#      문제로 정리
#
#
# 5. 전체 회귀 검증 (cell 68~75)
#    - 위 모든 변경사항을 core40, rag-56, set-13(Precision/Recall/F1)
#      전체로 다시 돌려서 부작용 없이 개선됐는지 최종 확인
# ============================================

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/sprint-public-procurement-rag-assistant
!pwd

/content/sprint-public-procurement-rag-assistant
/content/sprint-public-procurement-rag-assistant


In [3]:
import sys
import types
import src.data_processing.chunking as real_chunking

import pickle

from src.retrieval.indexing import HybridIndex
from src.data_processing.chunking import Chunk

import src.config as config
from pathlib import Path
import src.retrieval.indexing as indexing_module

config.CHROMA_DIR = Path('/content/drive/MyDrive/중급 프로젝트/chroma_db')

indexing_module.CHROMA_DIR = config.CHROMA_DIR
chunking_alias = types.ModuleType('src.chunking')
chunking_alias.Chunk = real_chunking.Chunk
sys.modules['src.chunking'] = chunking_alias

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')
with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)

index = HybridIndex(chunks)

[HybridIndex] parent 전략 chunk 3664개는 검색 후보에서 제외(context 확장 조회 전용) - 실제 검색 대상 14575개
[embeddings] SentenceTransformer 모델 로드 시도 중... (처음 실행이면 HuggingFace에서 모델을 내려받아 몇 분 걸릴 수 있습니다)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

[embeddings] SentenceTransformer 사용: nlpai-lab/KURE-v1 (dim=1024)
[HybridIndex] 기존 임베딩 인덱스 재사용: output/chroma_db (collection=rfp_chunks__nlpai-lab_KURE-v1, backend=nlpai-lab/KURE-v1, 검색 대상 chunk 14575개 일치, 재임베딩 건너뜀)


In [4]:
child_chunks = index._searchable_chunks
print(f"검색 대상(child) chunk 수: {len(child_chunks)}")

검색 대상(child) chunk 수: 14575


In [5]:
from google.colab import userdata
import openai

api_key = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=api_key)

In [6]:
seen = set()
all_filenames_with_biz = []
for c in child_chunks:
    if c.doc_id not in seen:
        seen.add(c.doc_id)
        biz = c.metadata.get('발주_기관', '')
        all_filenames_with_biz.append((c.doc_id, biz))

print(f"고유 문서 수: {len(all_filenames_with_biz)}")

고유 문서 수: 98


In [7]:
import sys as sys2
from generation_prompts import SYSTEM_PROMPT_V9, METADATA_DISTINCTION_INSTRUCTION, needs_metadata_distinction
from answer_generation import ask_rfp_v9, extract_doc_hints_multi, find_relevant_keywords

sys2.path.append('/content/drive/MyDrive/중급 프로젝트')

print("import 성공")

import 성공


In [8]:
import json, re
from src.generation.generation import check_required_facts

DATA_DIR2 = Path('/content/drive/MyDrive/중급 프로젝트')
with open(DATA_DIR2 / 'dev.refined.review-candidate.jsonl', encoding='utf-8') as f:
    core40 = [json.loads(l) for l in f if l.strip()]
with open(DATA_DIR2 / 'rag-56.draft.jsonl', encoding='utf-8') as f:
    rag56 = [json.loads(l) for l in f if l.strip()]
print(f"core40: {len(core40)}개, rag56: {len(rag56)}개")

core40: 40개, rag56: 56개


In [9]:
idx = SYSTEM_PROMPT_V9.find("5. 답변 끝에는")
print(SYSTEM_PROMPT_V9[idx:idx+100])

5. 답변 끝에는 근거가 된 문서명을 명시해.

## 답변을 거절/기권해야 하는 경우 (매우 중요)

아래 경우에는 문서 안에서 관련 정보를 억지로 찾아서 답하려 하지 말고, 명확


In [10]:
with open('/content/drive/MyDrive/중급 프로젝트/generation_prompts.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "5. 답변 끝에는 근거가 된 문서명을 명시해."
new_code = """5. 답변 끝에는 반드시 다음 형식으로 근거 문서를 명시해: [근거: 파일명1.hwp, 파일명2.hwp]
   대괄호와 콤마 형식을 정확히 지켜야 하며, 파일명은 컨텍스트에 표시된 문서명(확장자 포함)을
   원본 그대로 사용해. 인용한 문서가 하나뿐이어도 대괄호 형식은 유지해."""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/generation_prompts.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [11]:
with open('/content/drive/MyDrive/중급 프로젝트/generation_prompts.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """5. 답변 끝에는 반드시 다음 형식으로 근거 문서를 명시해: [근거: 파일명1.hwp, 파일명2.hwp]
   대괄호와 콤마 형식을 정확히 지켜야 하며, 파일명은 컨텍스트에 표시된 문서명(확장자 포함)을
   원본 그대로 사용해. 인용한 문서가 하나뿐이어도 대괄호 형식은 유지해."""

new_code = """5. 답변 끝에는 반드시 아래의 정확한 형식으로만 근거 문서를 표기해야 해. 다른 표현(예: "근거 문서:", "출처:")은 절대 쓰지 마:
   [근거: 문서명1, 문서명2]
   - 대괄호 [ ]를 반드시 포함하고, "근거:"라는 단어를 정확히 써야 해.
   - 문서명은 컨텍스트에 표시된 파일명(확장자 포함)을 원본 그대로 사용해.
   - 근거 문서가 하나뿐이어도 대괄호 형식은 그대로 유지해: [근거: 문서명1]
   - 여러 문서를 인용할 때는 콤마(,)로 구분해."""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/generation_prompts.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [13]:
import importlib
import generation_prompts
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(generation_prompts)
importlib.reload(answer_generation)


test_qs = [
    "BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?",
    "APC-HUB 고도화 2차사업과 NCIC 시스템 운영·개선 사업 중 예산이 더 큰 것은 무엇이고 차이는 얼마인가요?",
]

for q in test_qs:
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

사업예산: 243,000,000원 — 부가가치세(VAT) 포함되어 있습니다.  
[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

- APC-HUB 고도화 2차사업: 35,750,000원 (부가가치세 포함)
- NCIC 시스템 운영·개선 사업: 50,000,000원 (VAT 포함)

결론: NCIC 사업이 더 큽니다. 차이는 14,250,000원이다.

[근거: 대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp, 한국교육과정평가원_국가교육과정정보센터(NCIC) 시스템 운영 및 개선.hwp]



In [14]:
final_citation_check_40 = []
for item in core40:
    task_type = item['task_type']
    question = item['question']
    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_citation_check_40.append({'case_id': item['case_id'], 'task_type': task_type, 'answer': answer})
    print(f"[{item['case_id']}][{task_type}] {question}")
    print(answer)
    print()

[dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 243,000,000원 — 부가가치세(VAT) 포함으로 표기되어 있습니다.  
[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)  
예산: 70,000,000원(금칠천만원, VAT 포함)

[근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp]

[dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 1차 사업(시스템 구축 및 초기 데이터 구축)과 2차 사업(리포팅툴 및 리포트 출력양식 개발)으로 총 2차로 나뉩니다.
- 평가 비중은 기술평가 90%, 가격평가 10%입니다.

[근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp]

[dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
사업기간: 계약일로부터 90일(계약일 기준)  
시범 도입 규모: 1단계 시범도입 3개 기관(서울 2개소, 울산 1개소)  
[근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp]

[dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출방법: 입찰서와 제안서를 모두 전자적으로 나라장터(e-발주시스템)를 통해 제출해야 함(입찰서는 전자 제출만 허용). 제출기한은 입찰서 제출기한과 동일.  
- 파일 형식: 나라장터를 통해 제출하는 제안서류 일체는 PDF 파일 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
이 기능은 제가 수행할 수 없습니다. 실시간 정보는 제공된 문서에서 확인할 수 없습니다.  
[근거: 한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp, 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp, 한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp, 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp, 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp, 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp]

[dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다
[근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp)]

[dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
문서만으로 귀사의 자격 충족 여부를 판단할 수 없습니다.

필요한 조건(문서에 명시된 입찰참가자격) — 아래 항목들을 귀사 자료와 대조해 직접 확인해 주세요:
1. 부정당업자 해당 여부(지방자치단체 계약법 시행령 제92조 해당 여부)  
2. 주된 영업소 소재지: 입찰공고일 전일부터 계약체결일까지 부산광역시 소재 여부(지방자치단체 계약법 시행령 제20조 제1항 6호·동법 시행규칙 제24조 제2항)  
3. 나라장터(G2B)에서 소프트웨어사업자(업종코드 1468)로 입찰참가자격 등록 여부(입찰서 제출마감일 전일까지)  
4. 대기업·중견기업(소프트웨어산업 진흥법 제48조 등) 또는 상호출자제한기업집단 소속여부(문서상 참여불가)  
5. 중소기업제품 직접생산확인증명서(세부품명번호 8111159901) 보유 여부(입찰마감 전일까지 발급·유효)  
6. 공동수급·하도급 금지(공동수급 및 하도급 허용되지 않음) 준수 여부

위 항목들 중 귀사에서 확인 가능한 각 항목의 증빙을 준비하면, 문서 기준으로 자격 충족 여부를 판정하는 데 필요한 근거가 됩니다.  
이 기능(문서와 귀사 정보를 대조해 최종 판정)은 제가 수행할 수 없습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[dev-unknown-005][unknown] 서민금융 채팅상담 사업에 경쟁사가 몇 곳 참여할 예정이야?
확인되지 않습니다.

[근거: 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp]

[dev-unknown-006][unknown] 인천해양박물관 시스템

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
확인되지 않습니다.

[근거: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp, 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp]

[dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
요청하신 "수주확률을 %로 계산"은 제안요청서 문서만으로는 판정·확률 계산이 불가합니다. 따라서 답변할 수 없습니다.

판단에 필요한 주요 조건(간단 안내)
- 과거 수주 이력 및 평가점수(발주처별)
- 제출 가능한 제안서 가격(희망 낙찰가) 및 예산 대비 경쟁력
- 요구 인력·기술(예: 파견인원, PM 경력, 보유기술)의 보유 현황
- 공공입찰 관련 인증·등록(나라장터 등) 및 지역 제한 충족 여부
- 제안서의 기술·가격 배점 예상 및 경쟁사 현황
- 일정(파견·개발 가능 기간) 및 긴급공고/재공고 여부

위 항목들이 있어야만 상대적 수주 가능성을 평가할 수 있습니다. 문서 내 근거로 판정할 수 있는 수치·정보는 없습니다.  
[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp, 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
문서에 없는 정보는 임의로 가정해서 확정할 수 없습니다
[근거: 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp]

[dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
실시간 진행률 및 계약 지연 가능성은 제공된 문서에서 확인할 수 없습니다. 이 기능은 제가 수행할 수 없습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



In [15]:
def normalize_text(t):
    return t.replace(',', '').replace(' ', '')

def normalize_dates(text):
    text = re.sub(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', r'\1.\2.\3', text)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', lambda m: f"{m.group(1)}.{int(m.group(2)):02d}.{int(m.group(3)):02d}", text)
    return text

def _text_included(fact_text, answer_text):
    fact_text = normalize_dates(fact_text)
    answer_text_n = normalize_dates(answer_text)
    answer_norm = normalize_text(answer_text_n)

    numbers = re.findall(r'\d+(?:\.\d+)?', fact_text)
    numbers = [n for n in numbers if len(n) >= 2]

    if numbers:
        all_numbers_match = True
        for num in numbers:
            if num in answer_norm:
                continue
            if len(num) == 4 and num.startswith('20'):
                if num[2:] in answer_norm:
                    continue
            num_no_zero = re.sub(r'^0+', '', num)
            if num_no_zero and num_no_zero in answer_norm:
                continue
            all_numbers_match = False
            break

        if not all_numbers_match:
            return False

        raw_words = re.split(r'[\s,·:()]+', fact_text)
        stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
        core_words = []
        for w in raw_words:
            w = w.rstrip('.,')
            if len(w) < 2:
                continue
            if re.match(r'^\d', w):
                continue
            for suf in stopwords_suffix:
                if w.endswith(suf) and len(w) > len(suf):
                    w = w[:-len(suf)]
                    break
            w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
            if len(w) >= 2:
                core_words.append(w)

        if not core_words:
            return True

        match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
        return match_count / max(len(core_words), 1) >= 0.2

    raw_words = re.split(r'[\s,·:()]+', fact_text)
    stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
    core_words = []
    for w in raw_words:
        w = w.rstrip('.,')
        if len(w) < 2:
            continue
        if re.match(r'^\d', w):
            continue
        for suf in stopwords_suffix:
            if w.endswith(suf) and len(w) > len(suf):
                w = w[:-len(suf)]
                break
        w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
        if len(w) >= 2:
            core_words.append(w)

    if not core_words:
        return True

    match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
    return match_count / max(len(core_words), 1) >= 0.4

ABSTAIN_PHRASES = ['확인되지 않습니다', '답변할 수 없', '수행할 수 없', '확인할 수 없', '판단할 수 없',
                    '판정할 수 없', '판정해줄 수 없', '계산할 수 없', '제공할 수 없', '받아들일 수 없',
                    '확정할 수 없', '알려드릴 수 없', '불가능합니다', '제공된 문서 범위에서는',
                    '해드릴 수 없', '드릴 수 없']

def official_score_core40(item, answer_text):
    gold = item['gold']
    decision = gold.get('decision')

    if decision == 'abstain':
        is_abstained = any(p in answer_text for p in ABSTAIN_PHRASES)
        return 100 if is_abstained else 0

    key_points = gold.get('required_key_points', [])
    if not key_points:
        return None

    included = [_text_included(kp['text'], answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

In [16]:
for r in final_citation_check_40:
    item = next(it for it in core40 if it['case_id'] == r['case_id'])
    r['score'] = official_score_core40(item, r['answer'])
    print(f"[{r['case_id']}][{r['task_type']}] 점수: {r['score']}")

valid_scores = [r['score'] for r in final_citation_check_40 if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

[dev-single-001][single_doc] 점수: 100.0
[dev-single-002][single_doc] 점수: 100.0
[dev-single-003][single_doc] 점수: 100.0
[dev-single-004][single_doc] 점수: 100.0
[dev-single-005][single_doc] 점수: 100.0
[dev-single-006][single_doc] 점수: 100.0
[dev-single-007][single_doc] 점수: 100.0
[dev-single-008][single_doc] 점수: 100.0
[dev-single-009][single_doc] 점수: 100.0
[dev-single-010][single_doc] 점수: 66.67
[dev-multi-001][multi_doc_compare] 점수: 100.0
[dev-multi-002][multi_doc_compare] 점수: 100.0
[dev-multi-003][multi_doc_compare] 점수: 100.0
[dev-multi-004][multi_doc_compare] 점수: 100.0
[dev-multi-005][multi_doc_compare] 점수: 75.0
[dev-multi-006][multi_doc_compare] 점수: 100.0
[dev-multi-007][multi_doc_compare] 점수: 100.0
[dev-multi-008][multi_doc_compare] 점수: 100.0
[dev-multi-009][multi_doc_compare] 점수: 100.0
[dev-multi-010][multi_doc_compare] 점수: 50.0
[dev-followup-001][follow_up] 점수: 100.0
[dev-followup-002][follow_up] 점수: 50.0
[dev-followup-003][follow_up] 점수: 100.0
[dev-followup-004][follow_up] 점수: 100.0
[de

In [17]:
final_citation_check_56 = []
for item in rag56:
    question = item['question']
    answer = ask_rfp_v9(question, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_citation_check_56.append({'case_id': item['case_id'], 'answer': answer})
    print(f"[{item['case_id']}] {question}")
    print(answer)
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c01] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
999,494,600원(부가세 포함)  
[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c02] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024. 10. 31.까지 완료해야 합니다.  
[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[supplemental-qa-c03] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
1,515,000,000원 (사업예산: 1,515,000천원, 부가세 포함)

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[supplemental-qa-c04] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
입찰 방식: 제한경쟁입찰  
낙찰(사업자 선정) 절차: 협상에 의한 계약(협상으로 사업자 선정)

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[supplemental-qa-c05] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월간 수행합니다.  
[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[supplemental-qa-c06] 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성한 예산은 얼마인가요? 세금 포함 여부도 알려주세요.
사업예산은 181,913,000원이며 VAT 포함으로 표기되어 있습니다.
[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[supplemental-qa-c07] 인천공항운영서비스가 회계·인사 등 경영업무를 통합할 차세대 시스템을 구축하는 데 걸리는 기간은 얼마인가요?
계약 체결일로부터 9개월(안정화 기간 포함)입니다.

[근거: 인천공항운영서비스(주)_인천공항운영서

In [18]:
for r in final_citation_check_56:
    item = next(it for it in rag56 if it['case_id'] == r['case_id'])
    matched, total = check_required_facts(r['answer'], item['gold'].get('required_fact_groups'))
    r['score'] = round(matched / total * 100, 2) if total else None
    print(f"[{r['case_id']}] 점수: {r['score']}")

valid_scores_56 = [r['score'] for r in final_citation_check_56 if r['score'] is not None]
print(f"\nrag-56 전체 평균: {sum(valid_scores_56)/len(valid_scores_56):.2f}/100 ({len(valid_scores_56)}개)")

[supplemental-qa-c01] 점수: 100.0
[supplemental-qa-c02] 점수: 100.0
[supplemental-qa-c03] 점수: 100.0
[supplemental-qa-c04] 점수: 100.0
[supplemental-qa-c05] 점수: 100.0
[supplemental-qa-c06] 점수: 100.0
[supplemental-qa-c07] 점수: 100.0
[supplemental-qa-c08] 점수: 100.0
[supplemental-qa-c09] 점수: 100.0
[supplemental-qa-c10] 점수: 100.0
[supplemental-qa-c11] 점수: 100.0
[supplemental-qa-c12] 점수: 66.67
[supplemental-qa-c13] 점수: 100.0
[supplemental-qa-c14] 점수: 100.0
[supplemental-qa-c15] 점수: 100.0
[supplemental-qa-c16] 점수: 0.0
[supplemental-qa-c18] 점수: 100.0
[supplemental-qa-c19] 점수: 33.33
[supplemental-qa-c20] 점수: 50.0
[supplemental-qa-c23] 점수: 100.0
[supplemental-qa-c25] 점수: 50.0
[supplemental-qa-g01] 점수: 100.0
[supplemental-qa-g02] 점수: 100.0
[supplemental-qa-g03] 점수: 50.0
[supplemental-qa-g04] 점수: 100.0
[supplemental-qa-g05] 점수: 100.0
[supplemental-qa-g06] 점수: 100.0
[supplemental-qa-g07] 점수: 100.0
[supplemental-qa-g08] 점수: 80.0
[supplemental-qa-g11] 점수: 33.33
[supplemental-qa-g12] 점수: 75.0
[supplemental-q

In [19]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = '''            return f"'{fname}' 사업입니다. 예산은 {amt:,.0f}원이며, 차이는 {diff:,.0f}원입니다.\\n\\n근거: {fname} (사업금액 메타데이터 기준)"'''
new_code = '''            return f"'{fname}' 사업입니다. 예산은 {amt:,.0f}원이며, 차이는 {diff:,.0f}원입니다.\\n\\n[근거: {fname}]"'''

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [20]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(answer_generation)

q_g25 = "데이터셋에서 사업명에 '구축'이 포함된 사업만 대상으로 할 때, 서민금융 채팅 상담시스템 구축 사업(230,000,000원)과 예산 차이가 가장 작은 다른 사업은 무엇인가요?"
answer = ask_rfp_v9(q_g25, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

'한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp' 사업입니다. 예산은 212,300,000원이며, 차이는 17,700,000원입니다.

[근거: 한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp]


In [22]:
with open('/content/sprint-public-procurement-rag-assistant/src/evaluation/golden_set_v3.py', 'r', encoding='utf-8') as f:
    print(f.read())

"""golden-set-v3-share 패키지(팀원이 2026-09-01에 공유한 3차 API 기준선/보조
골든셋 묶음)를 우리 파이프라인이 쓸 수 있는 형태로 불러온다.

[배경] 팀원이 Downloads/golden-set-v3-share 폴더를 공유했다. 패키지 자체는
"최종 평가 자산 131건"(기존 유지 111 + 신규 20)을 표방하지만, 실제로 우리
코퍼스(output/merged_docs.pkl)와 하나씩 대조해보니 그대로 다 쓸 수는
없었다:

1. **lane(문항 그룹)마다 우리 코퍼스와 매칭 가능한 정보량이 다르다.**
   - `rag-56`(답변형 56건) / `set-13`(집합검색형 13건): `source_labels`
     필드에 사람이 읽을 수 있는 원본 파일명이 있어서 우리 코퍼스 doc_id와
     매칭 가능. 단 공백 2칸 vs 1칸 차이가 있는 파일명이 몇 개 있어서
     공백을 정규화해야 정확히 일치한다(예: "그랜드코리아레저(주)_2024년도
     GKL  그룹웨어..." 처럼 우리 코퍼스 쪽에 공백이 2칸인 경우가 있음).
   - `visual`(표/그림 10건): `document.source_filename`에 파일명이 있지만
     전부 "refined_" 접두어가 붙어있다. 접두어를 떼면 우리 코퍼스 doc_id와
     정확히 일치.
   - **`core40`(기존 핵심 40건)과 `corpus_analytics`(전체 통계 10건, 총
     50건)는 이 모듈이 읽지 않는다.** doc_id가 "doc_e3b910313338c8c5232ec2de"
     같은 내부 해시/ID뿐이고, 사람이 읽을 수 있는 파일명이나 우리 코퍼스에
     대조할 매핑 매니페스트가 이번 공유 패키지에 없었다(README가 "raw RFP
     source files"는 제외했다고 밝혔는데 이 doc_id->파일명 매핑 파일도 같이
     빠진 것으로 보임) - 팀원한테 매핑 매니페스트를 받

In [28]:
import os
import shutil

src_dir = '/content/drive/MyDrive/중급 프로젝트'
dst_dir = '/content/sprint-public-procurement-rag-assistant/data/golden_set_v3'
os.makedirs(dst_dir, exist_ok=True)

files_to_copy = ['rag-56.draft.jsonl', 'set-13.draft.jsonl', 'document-structure-visual-qa.jsonl']
for fname in files_to_copy:
    src = os.path.join(src_dir, fname)
    dst = os.path.join(dst_dir, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"복사 완료: {fname}")
    else:
        print(f"없음: {fname}")

복사 완료: rag-56.draft.jsonl
복사 완료: set-13.draft.jsonl
복사 완료: document-structure-visual-qa.jsonl


In [29]:
from src.evaluation.golden_set_v3 import load_golden_set_v3

corpus_doc_ids = set(all_filenames_with_biz_dict := dict(all_filenames_with_biz)) if False else {fname for fname, _ in all_filenames_with_biz}

golden_v3 = load_golden_set_v3(corpus_doc_ids=corpus_doc_ids)
print(golden_v3.shape)
print(golden_v3['source_lane'].value_counts())

[load_golden_set_v3] 79건 로드(answer/visual 66건 + set 13건). 원본 패키지의 core40(40)/corpus_analytics(10) 총 50건은 우리 코퍼스와 매칭할 방법이 없어서 제외.
[load_golden_set_v3] 참고: enabled=False 69건, review.status=draft 79건 (패키지 자체가 아직 팀 승인 전이라고 명시한 항목들 - 그래도 그대로 평가에 포함시켰음, v3_enabled/v3_review_status 컬럼으로 나중에 필터링 가능)
(79, 11)
source_lane
answer    56
set       13
visual    10
Name: count, dtype: int64


In [30]:
new_items = golden_v3[golden_v3['source_lane'].isin(['set', 'visual'])]
print(f"새로 확인할 문항 수: {len(new_items)}개\n")
for _, row in new_items.iterrows():
    print(f"[{row['id']}] ({row['source_lane']}) {row['query']}")

새로 확인할 문항 수: 23개

[supplemental-set-b1] (set) 철도 시설을 가상 공간에 구현하기 위한 디지털 전환 전략을 수립하는 공공기관 사업은 무엇인가요?
[supplemental-set-b10] (set) 코레일이 승차권 예매·발매 플랫폼의 개편 방향을 세우기 위해 진행한 계획수립 용역은 무엇인가요?
[supplemental-set-b12] (set) 학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘
[supplemental-set-b14] (set) 긴급으로 진행되는 사업 알려줘
[supplemental-set-b15] (set) 3개월 이내로 끝나는 짧은 사업 있어?
[supplemental-set-b16] (set) 보안 관련 시스템 구축 사업 알려줘
[supplemental-set-b20] (set) 한국원자력연구원 선량평가시스템 사업이 왜 추진되는지 목적을 알려줘
[supplemental-set-b21] (set) 재난 관련 사업 중 예산이 5억 이상인 것만 알려줘
[supplemental-set-b22] (set) 학교 발주 사업 중에서 예산이 가장 큰 곳은?
[supplemental-set-b23] (set) 봉화군 재난통합관리시스템과 충북연구원 재난안전데이터 사업의 공통점과 차이점은?
[supplemental-set-b24] (set) 지방자치단체가 발주한 사업 중 예산이 1억 이상인 것만 알려줘
[supplemental-set-b3] (set) 병원 전산 장애나 재해에 대비해 진료정보를 복구할 체계를 마련하는 적십자 관련 용역은 무엇인가요?
[supplemental-set-b4] (set) 학생들의 교과 밖 활동을 관리할 온라인 환경을 새로 만드는 을지대 사업은 무엇인가요?
[visual-hwp-table-001] (visual) 수문자료정보관리시스템(HDIMS) 재구축 용역(3단계)의 기술성 평가 구성표에서 정량적 평가와 정성적 평가의 소계 및 평가 방식은 각각 무엇인가?
[visual-hwp-table-0

In [31]:
set_test_questions = [
    "학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘",
    "긴급으로 진행되는 사업 알려줘",
    "3개월 이내로 끝나는 짧은 사업 있어?",
    "재난 관련 사업 중 예산이 5억 이상인 것만 알려줘",
    "학교 발주 사업 중에서 예산이 가장 큰 곳은?",
    "지방자치단체가 발주한 사업 중 예산이 1억 이상인 것만 알려줘",
]

for q in set_test_questions:
    print(f"=== {q} ===")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

=== 학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘 ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

- 광주과학기술원 — 학사시스템 기능개선 사업; 사업예산: 157,300,000원(VAT 포함).  
- 고려대학교 — 차세대 포털·학사 정보시스템 구축사업; 사업예산: 11,270,000,000원.  
- 서영대학교 산학협력단(서영대학교) — 서영대학교 차세대 교육혁신지원시스템 3단계 구축 사업; 사업예산: 950,000,000원(VAT 포함).  
- 을지대학교 — 을지대학교 비교과시스템 개발 사업; 사업예산: 비공개.  
- 서울시립대학교 — 학업성취도 다차원 종단분석 통합시스템(1차); 사업예산: 확인되지 않습니다.

[근거: 광주과학기술원_학사시스템 기능개선 사업.hwp, 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf, 서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp, 을지대학교_을지대학교 비교과시스템 개발.hwp, 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf]

=== 긴급으로 진행되는 사업 알려줘 ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

긴급으로 진행되는 사업 목록(문서 기준)

- 국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp — 문서 내에 '긴급입찰 사유서' 및 긴급입찰 근거 설명(국가계약법 시행령 등) 포함.  
- 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp — 문서에 '긴급공고', '긴급발주' 등 긴급 추진 사유 및 일정 명시.  
- 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp — 파일명에 '(긴급)' 표기 및 사업 개요(급박한 응급의료 대응 필요) 포함.  
- 한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp — 파일명에 '(긴급)' 표기.

[근거: 국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp, 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp, 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp, 한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp]

=== 3개월 이내로 끝나는 짧은 사업 있어? ===


...

- 봉화군 재난통합관리시스템 고도화 사업 — 900,000,000원(부가세 포함)  
- 국립중앙의료원 차세대 응급의료 상황관리시스템 구축 — 1,400,000,000원(메타데이터)  
- 한국산업단지공단 산단 안전정보시스템 1차 구축 용역 — 717,285,200원(부가세 포함)  
- 적십자병원 병원정보 재해복구시스템 구축 — 500,000,000원(부가세 포함)

[근거: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp, 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp, 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp, 대한적십자사 의료원_적십자병원 병원정보 재해복구시스템 구축 용역 .hwp]

=== 학교 발주 사업 중에서 예산이 가장 큰 곳은? ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

고려대학교_차세대 포털·학사 정보시스템 구축사업 — 11,270,000,000원

[근거: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf]

=== 지방자치단체가 발주한 사업 중 예산이 1억 이상인 것만 알려줘 ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

다음 문서는 발주기관(지방자치단체) 사업이며 사업금액이 100,000,000원(1억) 이상입니다.

- 인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp — 발주기관: 인천광역시 — 사업금액: 150,000,000원  
- 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp — 발주기관: 전북특별자치도 정읍시 — 사업금액: 200,000,000원  
- 울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp — 발주기관: 울산광역시 — 사업금액: 986,945,000원  
- 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp — 발주기관: 경기도 평택시 — 사업금액: 999,494,600원  
- 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp — 발주기관: 인천광역시 — 사업금액: 100,000,000원  
- 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp — 발주기관: 인천광역시 동구 — 사업금액: 200,000,000원  
- 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp — 발주기관: 경상북도 봉화군 — 사업금액: 900,000,000원  
- 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf — 발주기관: 서울특별시 — 사업금액: 493,763,000원

[근거: 인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp, 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp, 울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp, 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp, 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp, 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp, 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp, 서울특별시_2024년 지도정보 플랫폼

In [32]:
set_items = golden_v3[golden_v3['source_lane'] == 'set']

for _, row in set_items.iterrows():
    print(f"[{row['id']}] {row['query']}")
    print(f"  정답 문서 목록({len(row['expected_doc_id'])}개): {row['expected_doc_id']}")
    print()

[supplemental-set-b1] 철도 시설을 가상 공간에 구현하기 위한 디지털 전환 전략을 수립하는 공공기관 사업은 무엇인가요?
  정답 문서 목록(1개): ['국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp']

[supplemental-set-b10] 코레일이 승차권 예매·발매 플랫폼의 개편 방향을 세우기 위해 진행한 계획수립 용역은 무엇인가요?
  정답 문서 목록(1개): ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp']

[supplemental-set-b12] 학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘
  정답 문서 목록(12개): ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp', '고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', '서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf', '경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp', '전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp', '광주과학기술원_학사시스템 기능개선 사업.hwp', '을지대학교_을지대학교 비교과시스템 개발.hwp', '대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp', '서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp', '광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp', '남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp', '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp']

[supplemental-set-b14] 긴급으로 진행되는 사업 알려줘
  정답 문서 목록(5개): ['KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp', '경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사

In [34]:
from answer_generation import is_aggregation_question, extract_filter_conditions

q_b12 = "학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘"
print("집계 질문 감지:", is_aggregation_question(q_b12))
print("필터 조건:", extract_filter_conditions(q_b12))

hints = extract_doc_hints_multi(q_b12, all_filenames_with_biz)
print("문서 힌트:", hints)

집계 질문 감지: False
필터 조건: {}
문서 힌트: []


In [35]:
# 학교 필터 조건 추가 및 테스트
def is_school_org(org):
    """발주기관명이 학교(대학교/대학/과학기술원 등)인지 판별"""
    if org is None or (isinstance(org, float)):
        return False
    school_keywords = ['대학교', '대학', '과학기술원', '산학협력단']
    return any(kw in str(org) for kw in school_keywords)

# 전체 문서에서 학교로 분류되는 것들 확인
school_docs = [(fname, biz) for fname, biz in all_filenames_with_biz if is_school_org(biz)]
print(f"학교로 분류된 문서 수: {len(school_docs)}개")
for fname, biz in school_docs:
    print(f"  {biz}: {fname}")

학교로 분류된 문서 수: 13개
  한영대학: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
  고려대학교: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
  (사）한국대학스포츠협의회: (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp
  서울시립대학교: 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf
  경희대학교: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
  전북대학교: 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp
  광주과학기술원: 광주과학기술원_학사시스템 기능개선 사업.hwp
  을지대학교: 을지대학교_을지대학교 비교과시스템 개발.hwp
  대전대학교: 대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp
  서영대학교 산학협력단: 서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp
  광주과학기술원: 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp
  남서울대학교: 남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp
  조선대학교: 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp


In [36]:
def is_school_org(org):
    """발주기관명이 학교(대학교/대학/과학기술원 등)인지 판별.
    '대학스포츠협의회' 같은 협회는 제외."""
    if org is None or (isinstance(org, float)):
        return False
    org_str = str(org)
    if '협의회' in org_str or '협회' in org_str:
        return False
    school_keywords = ['대학교', '대학', '과학기술원']
    return any(kw in org_str for kw in school_keywords)

school_docs_v2 = [(fname, biz) for fname, biz in all_filenames_with_biz if is_school_org(biz)]
print(f"학교로 분류된 문서 수: {len(school_docs_v2)}개")
for fname, biz in school_docs_v2:
    print(f"  {biz}")

학교로 분류된 문서 수: 12개
  한영대학
  고려대학교
  서울시립대학교
  경희대학교
  전북대학교
  광주과학기술원
  을지대학교
  대전대학교
  서영대학교 산학협력단
  광주과학기술원
  남서울대학교
  조선대학교


In [37]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """def extract_filter_conditions(query):
    conditions = {}
    if '억' in query and ('이상' in query or '넘는' in query):
        match = re.search(r'(\\d+)억', query)
        if match:
            conditions['금액_최소'] = int(match.group(1)) * 100000000
    if '지자체' in query or '지방자치단체' in query:
        conditions['지자체'] = True
    if '공사' in query and ('OO공사' in query or '발주기관이' in query):
        conditions['공사'] = True
    if 'AI' in query:
        conditions['주제_AI'] = True
    if '긴급' in query:
        conditions['긴급'] = True
    if '보안' in query:
        conditions['보안'] = True
    if '재난' in query:
        conditions['재난'] = True
    return conditions"""

new_code = """def is_school_org(org):
    \"\"\"발주기관명이 학교(대학교/대학/과학기술원 등)인지 판별.
    '대학스포츠협의회' 같은 협회는 제외.\"\"\"
    if org is None or (isinstance(org, float)):
        return False
    org_str = str(org)
    if '협의회' in org_str or '협회' in org_str:
        return False
    school_keywords = ['대학교', '대학', '과학기술원']
    return any(kw in org_str for kw in school_keywords)


def extract_filter_conditions(query):
    conditions = {}
    if '억' in query and ('이상' in query or '넘는' in query):
        match = re.search(r'(\\d+)억', query)
        if match:
            conditions['금액_최소'] = int(match.group(1)) * 100000000
    if '지자체' in query or '지방자치단체' in query:
        conditions['지자체'] = True
    if '공사' in query and ('OO공사' in query or '발주기관이' in query):
        conditions['공사'] = True
    if 'AI' in query:
        conditions['주제_AI'] = True
    if '긴급' in query:
        conditions['긴급'] = True
    if '보안' in query:
        conditions['보안'] = True
    if '재난' in query:
        conditions['재난'] = True
    if '학교' in query or '대학교' in query or '대학' in query or '과학기술원' in query:
        conditions['학교'] = True
    return conditions"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [38]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """    def build_meta_filter(conds):
        if not conds:
            return None
        def _filter(meta):
            if '금액_최소' in conds:
                amt = meta.get('사업_금액')
                if amt is None or amt < conds['금액_최소']:
                    return False
            if conds.get('지자체'):
                if not is_local_gov(meta.get('발주_기관')):
                    return False
            if conds.get('공사'):
                org = str(meta.get('발주_기관', ''))
                if '공사' not in org:
                    return False
            return True
        return _filter"""

new_code = """    def build_meta_filter(conds):
        if not conds:
            return None
        def _filter(meta):
            if '금액_최소' in conds:
                amt = meta.get('사업_금액')
                if amt is None or amt < conds['금액_최소']:
                    return False
            if conds.get('지자체'):
                if not is_local_gov(meta.get('발주_기관')):
                    return False
            if conds.get('공사'):
                org = str(meta.get('발주_기관', ''))
                if '공사' not in org:
                    return False
            if conds.get('학교'):
                if not is_school_org(meta.get('발주_기관')):
                    return False
            return True
        return _filter"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [39]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9, extract_filter_conditions

importlib.reload(answer_generation)

q_b12 = "학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘"
print("필터 조건:", extract_filter_conditions(q_b12))
print()
answer = ask_rfp_v9(q_b12, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

필터 조건: {'학교': True}



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

다음 문서들이 학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 제안요청서입니다:

- 광주과학기술원_학사시스템 기능개선 사업.hwp  
- 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp  
- 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf  
- 서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp  
- 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf  
- 을지대학교_을지대학교 비교과시스템 개발.hwp  
- 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp  
- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp  
- 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp  
- 남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp  
- 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp

[근거: 광주과학기술원_학사시스템 기능개선 사업.hwp, 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp, 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf, 서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp, 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf, 을지대학교_을지대학교 비교과시스템 개발.hwp, 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp, 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp, 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp, 남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활 성화(학사.hwp, 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp]


In [40]:
expected_b12 = {'한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp',
                '고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf',
                '서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf',
                '경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp',
                '전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp',
                '광주과학기술원_학사시스템 기능개선 사업.hwp',
                '을지대학교_을지대학교 비교과시스템 개발.hwp',
                '대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp',
                '서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp',
                '광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp',
                '남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp',
                '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp'}

answer_docs_b12 = {'광주과학기술원_학사시스템 기능개선 사업.hwp',
                    '광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp',
                    '고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf',
                    '서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp',
                    '서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf',
                    '을지대학교_을지대학교 비교과시스템 개발.hwp',
                    '전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp',
                    '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp',
                    '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp',
                    '남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp',
                    '경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp'}

print("빠진 문서:", expected_b12 - answer_docs_b12)

빠진 문서: {'대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp'}


In [41]:
# 대전대 문서가 필터 조건을 통과하는지 직접 확인
from answer_generation import is_school_org

doc_to_meta_test = {}
for c in child_chunks:
    if c.doc_id not in doc_to_meta_test:
        doc_to_meta_test[c.doc_id] = c.metadata

daejeon_meta = doc_to_meta_test.get('대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp')
print(daejeon_meta)
print("필터 통과 여부:", is_school_org(daejeon_meta.get('발주_기관')) if daejeon_meta else "메타데이터 없음")

{'발주_기관': '대전대학교', '사업_금액': 60000000.0, 'budget_unknown': False, '입찰_참여_마감일': '2024-12-09 11:00:00', '입찰참여마감일_결측': False, '파일형식': 'hwp', 'doc_type': 'plain_text', 'source': 'csv_fallback'}
필터 통과 여부: True


In [42]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = "hits = index.hybrid_search(question, k=80, meta_filter=meta_filter, expand_to_parent=True)"
new_code = "hits = index.hybrid_search(question, k=200, meta_filter=meta_filter, expand_to_parent=True)"

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [43]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(answer_generation)

q_b12 = "학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘"
answer = ask_rfp_v9(q_b12, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

다음은 컨텍스트에 포함된 학교(대학교/대학/과학기술원) 발주 시스템 사업 목록입니다. 기관명 — 사업명 — 사업금액(문서 표기 기준)

- 광주과학기술원 — 학사시스템 기능개선 사업 — 157,300,000원(VAT 포함)  
- 광주과학기술원 — 실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업 — 54,450,000원(명시)  
- 고려대학교 — 차세대 포털·학사 정보시스템 구축 사업 — 11,270,000,000원(VAT 포함)  
- 서영대학교 산학협력단 — 차세대 교육혁신지원시스템 3단계 구축 사업 — 950,000,000원(VAT 포함)  
- 을지대학교 — 을지대학교 비교과시스템 개발 사업 — 사업비: 비공개 (문서 표기)  
- 서울시립대학교 — 학업성취도 다차원 종단분석 통합시스템 1차 고도화 — 242,900,000원(VAT 포함)  
- 한영대학 — 특성화 맞춤형 교육환경 구축(트랙운영 학사정보시스템 고도화) — 130,000,000원(VAT 포함)  
- 전북대학교(전북지역혁신플랫폼) — JST 공유대학(원) xAPI기반 LRS시스템 구축 — 116,000,000원(VAT 포함)  
- 조선대학교 — (재공고) 2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구축 — 100,000,000원(명시)  
- 남서울대학교 — [혁신-국고] 스마트 정보시스템 활성화(학사행정 암호화) — 70,000,000원(VAT 포함)  
- 경희대학교 산학협력단 — 정보시스템 운영 용역업체 선정(인포21 운영) — 400,000,000원(총예산, VAT 포함)

[근거: 광주과학기술원_학사시스템 기능개선 사업.hwp, 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp, 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf, 서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp, 을지대학교_을지대학교 비교과시스템 개발.hwp, 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 

In [44]:
from answer_generation import is_school_org

hits = index.hybrid_search(q_b12, k=200, meta_filter=None, expand_to_parent=True)
daejeon_in_hits = [h for h in hits if '대전대학교' in h.doc_id]
print(f"필터 없이 검색했을 때 대전대 청크: {len(daejeon_in_hits)}개 / 전체 {len(hits)}개")

# meta_filter 적용해서 다시 확인
def test_filter(meta):
    return is_school_org(meta.get('발주_기관'))

hits_filtered = index.hybrid_search(q_b12, k=200, meta_filter=test_filter, expand_to_parent=True)
print(f"필터 적용 후 전체 청크 수: {len(hits_filtered)}")
daejeon_in_filtered = [h for h in hits_filtered if '대전대학교' in h.doc_id]
print(f"필터 적용 후 대전대 청크: {len(daejeon_in_filtered)}개")

unique_docs_filtered = set(h.doc_id for h in hits_filtered)
print(f"필터 적용 후 고유 문서 수: {len(unique_docs_filtered)}개")
print(unique_docs_filtered)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

필터 없이 검색했을 때 대전대 청크: 0개 / 전체 29개


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

필터 적용 후 전체 청크 수: 30
필터 적용 후 대전대 청크: 0개
필터 적용 후 고유 문서 수: 11개
{'서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp', '을지대학교_을지대학교 비교과시스템 개발.hwp', '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp', '경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp', '고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', '전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp', '남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp', '광주과학기술원_학사시스템 기능개선 사업.hwp', '서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf', '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp', '광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp'}


In [45]:
q_b12 = "학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘"

matching_docs = [(fname, biz) for fname, biz in all_filenames_with_biz if is_school_org(biz)]
print(f"메타데이터로 직접 찾은 학교 문서: {len(matching_docs)}개")
for fname, biz in matching_docs:
    print(f"  {fname}")

메타데이터로 직접 찾은 학교 문서: 12개
  한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
  고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
  서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf
  경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
  전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp
  광주과학기술원_학사시스템 기능개선 사업.hwp
  을지대학교_을지대학교 비교과시스템 개발.hwp
  대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp
  서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp
  광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp
  남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp
  조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp


In [46]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """    elif conditions:
        meta_filter = build_meta_filter(conditions)
        hits = index.hybrid_search(question, k=200, meta_filter=meta_filter, expand_to_parent=True)
        for h in hits:
            context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\\n{h.text}")"""

new_code = """    elif conditions:
        # 벡터 검색 대신, 조건에 맞는 모든 문서를 메타데이터에서 직접 필터링
        # (벡터 유사도 순위가 낮아 검색 결과에서 누락되는 문서를 방지)
        meta_filter = build_meta_filter(conditions)
        matching_docs = [fname for fname, biz in all_filenames_with_biz
                          if meta_filter(doc_to_meta.get(fname, {}))]
        for fname in matching_docs:
            doc_c = get_doc_chunks(fname)
            header = meta_header_from_metadata(fname, doc_to_meta.get(fname, {}))
            for c in doc_c[:3]:
                context_parts.append(f"{header}\\n{c.text}")"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [47]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(answer_generation)

q_b12 = "학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘"
answer = ask_rfp_v9(q_b12, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

다음 문서들은 학교(대학교/대학/과학기술원)가 발주한 시스템 관련 사업입니다. (기관명 — 사업명)

- 한영대학 — 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화  
  (파일: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp)

- 고려대학교 — 차세대 포털·학사 정보시스템 구축 사업  
  (파일: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf)

- 서울시립대학교 — 학업성취도 다차원 종단분석 통합시스템 1차 고도화  
  (파일: 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf)

- 경희대학교 — 산학협력단 정보시스템(인포21) 운영 용역업체 선정(정보시스템 운영/유지관리)  
  (파일: 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp)

- 전북대학교 — JST 공유대학(원) xAPI 기반 LRS 시스템 구축  
  (파일: 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp)

- 광주과학기술원(GIST) — 학사시스템 기능개선 사업  
  (파일: 광주과학기술원_학사시스템 기능개선 사업.hwp)

- 광주과학기술원(GIST) — 실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업  
  (파일: 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp)

- 을지대학교 — 비교과시스템 개발 사업  
  (파일: 을지대학교_을지대학교 비교과시스템 개발.hwp)

- 대전대학교 — 다층적 융합 학습경험 플랫폼(MILE) 구축  
  (파일: 대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp)

- 서영대학교 산학협력단 — 차세대 교육혁신지원시스템 3단계 구축 사업  
  (파일: 서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp)

- 남서울대학교 — [혁신-국고] 스마트 정보시스템 활성화(학사행정 암호화)  
  (파일: 남서

In [48]:
recheck_conditions = [
    ("b14", "긴급으로 진행되는 사업 알려줘"),
    ("b21", "재난 관련 사업 중 예산이 5억 이상인 것만 알려줘"),
    ("b24", "지방자치단체가 발주한 사업 중 예산이 1억 이상인 것만 알려줘"),
]

for label, q in recheck_conditions:
    print(f"[{label}] {q}")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

[b14] 긴급으로 진행되는 사업 알려줘
다음 문서들이 '긴급'으로 표시되어 있습니다.

- 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
- KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp
- 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp
- 한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스템 개량.hwp
- 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp (문서 본문에 "긴급공고" 표기)

[근거: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp, KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp, 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp, 한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스템 개량.hwp, 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp]

[b21] 재난 관련 사업 중 예산이 5억 이상인 것만 알려줘
- 봉화군 재난통합관리시스템 고도화 사업 — 예산 900,000,000원(부가세 포함), 발주기관: 경상북도 봉화군  
- 2024년도 차세대 응급의료 상황관리시스템 구축 — 예산 1,400,000,000원(부가세 포함), 발주기관: 국립중앙의료원  
- 산단 안전정보시스템 1차 구축 용역 — 예산 717,285,200원(부가세 포함), 발주기관: 한국산업단지공단  
- 적십자병원 병원정보 재해복구시스템 구축 — 예산 500,000,000원(부가세 포함), 발주기관: 대한적십자사 의료원

[근거: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp, 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp, 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp, 대한적십자사 의료원_적십자병원 병원정보 재해복구시스템

In [49]:
expected_b14 = {'KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp',
                '경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp',
                '국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp',
                '한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp',
                '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp'}

answer_b14 = {'경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp',
              'KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp',
              '국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp',
              '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스템 개량.hwp',
              '서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp'}

print("빠진 것:", expected_b14 - answer_b14)
print("잘못 포함된 것:", answer_b14 - expected_b14)

빠진 것: {'한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp', '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp'}
잘못 포함된 것: {'한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스템 개량.hwp', '서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp'}


In [12]:
doc_id_check = '한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp'
doc_c = [c for c in child_chunks if c.doc_id == doc_id_check]
meta = doc_c[0].metadata if doc_c else None
print("메타데이터:", meta)
print()
print("'긴급' 포함 여부(파일명):", '긴급' in doc_id_check)

메타데이터: {'발주_기관': '한국수출입은행', '사업_금액': 1247000000.0, 'budget_unknown': False, '입찰_참여_마감일': '2024-11-05 16:00:00', '입찰참여마감일_결측': False, '파일형식': 'hwp', 'doc_type': 'plain_text', 'source': 'raw_parsed'}

'긴급' 포함 여부(파일명): True


In [13]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

idx = content.find("def build_meta_filter")
print(content[idx:idx+1200])

def build_meta_filter(conds):
        if not conds:
            return None
        def _filter(meta):
            if '금액_최소' in conds:
                amt = meta.get('사업_금액')
                if amt is None or amt < conds['금액_최소']:
                    return False
            if conds.get('지자체'):
                if not is_local_gov(meta.get('발주_기관')):
                    return False
            if conds.get('공사'):
                org = str(meta.get('발주_기관', ''))
                if '공사' not in org:
                    return False
            if conds.get('학교'):
                if not is_school_org(meta.get('발주_기관')):
                    return False
            return True
        return _filter

    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0]

In [14]:
# 파일명 기준으로 필터링했을 때 몇 개가 정확히 잡히는지 확인
def is_urgent_doc(fname):
    return '긴급' in fname

urgent_docs = [fname for fname, biz in all_filenames_with_biz if is_urgent_doc(fname)]
print(f"'긴급' 포함 파일: {len(urgent_docs)}개")
for f in urgent_docs:
    print(f"  {f}")

'긴급' 포함 파일: 5개
  경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
  KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp
  국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp
  한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp
  한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp


In [15]:
idx2 = content.find("elif conditions:")
print(content[idx2:idx2+700])

elif conditions:
        # 벡터 검색 대신, 조건에 맞는 모든 문서를 메타데이터에서 직접 필터링
        # (벡터 유사도 순위가 낮아 검색 결과에서 누락되는 문서를 방지)
        meta_filter = build_meta_filter(conditions)
        matching_docs = [fname for fname, biz in all_filenames_with_biz
                          if meta_filter(doc_to_meta.get(fname, {}))]
        for fname in matching_docs:
            doc_c = get_doc_chunks(fname)
            header = meta_header_from_metadata(fname, doc_to_meta.get(fname, {}))
            for c in doc_c[:3]:
                context_parts.append(f"{header}\n{c.text}")

    else:
        hits = index.hybrid_search(question, k=10, expand_to_parent=True)
        for h in hits:
            context_parts.append(f


In [16]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_filter_func = """    def build_meta_filter(conds):
        if not conds:
            return None
        def _filter(meta):
            if '금액_최소' in conds:
                amt = meta.get('사업_금액')
                if amt is None or amt < conds['금액_최소']:
                    return False
            if conds.get('지자체'):
                if not is_local_gov(meta.get('발주_기관')):
                    return False
            if conds.get('공사'):
                org = str(meta.get('발주_기관', ''))
                if '공사' not in org:
                    return False
            if conds.get('학교'):
                if not is_school_org(meta.get('발주_기관')):
                    return False
            return True
        return _filter"""

new_filter_func = """    def build_meta_filter(conds):
        if not conds:
            return None
        def _filter(meta, fname=''):
            if '금액_최소' in conds:
                amt = meta.get('사업_금액')
                if amt is None or amt < conds['금액_최소']:
                    return False
            if conds.get('지자체'):
                if not is_local_gov(meta.get('발주_기관')):
                    return False
            if conds.get('공사'):
                org = str(meta.get('발주_기관', ''))
                if '공사' not in org:
                    return False
            if conds.get('학교'):
                if not is_school_org(meta.get('발주_기관')):
                    return False
            if conds.get('긴급'):
                if '긴급' not in fname:
                    return False
            if conds.get('보안'):
                if '보안' not in fname:
                    return False
            if conds.get('재난'):
                if '재난' not in fname:
                    return False
            return True
        return _filter"""

count1 = content.count(old_filter_func)
print(f"filter func 찾은 개수: {count1}")
content = content.replace(old_filter_func, new_filter_func)

old_call = """        matching_docs = [fname for fname, biz in all_filenames_with_biz
                          if meta_filter(doc_to_meta.get(fname, {}))]"""
new_call = """        matching_docs = [fname for fname, biz in all_filenames_with_biz
                          if meta_filter(doc_to_meta.get(fname, {}), fname)]"""

count2 = content.count(old_call)
print(f"호출부 찾은 개수: {count2}")
content = content.replace(old_call, new_call)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

filter func 찾은 개수: 1
호출부 찾은 개수: 1
수정 완료


In [17]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(answer_generation)

recheck_v2 = [
    ("b14", "긴급으로 진행되는 사업 알려줘"),
    ("b16", "보안 관련 시스템 구축 사업 알려줘"),
    ("b21", "재난 관련 사업 중 예산이 5억 이상인 것만 알려줘"),
]

for label, q in recheck_v2:
    print(f"[{label}] {q}")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

[b14] 긴급으로 진행되는 사업 알려줘
다음 문서들이 '긴급'으로 표시된 사업입니다. (사업명 / 발주기관 / 사업금액)

- 봉화군 재난통합관리시스템 고도화 사업 / 경상북도 봉화군 / 900,000,000원(부가세 포함)  
- 우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축 및 지역의회 연계기반 개선 사업 PMC용역 / KOICA 전자조달 / 6,758,571,493원  
- 2024년도 차세대 응급의료 상황관리시스템 구축 / 국립중앙의료원 / 1,400,000,000원(부가세 포함)  
- 운행정보기록 자동분석시스템 개량 / 한국철도공사(용역) / 487,150,000원  
- 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업 타당성조사(F/S)용역 / 한국수출입은행 / 1,247,000,000원(부가세 포함)

[근거: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp, KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp, 국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp, 한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp, 한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp]

[b16] 보안 관련 시스템 구축 사업 알려줘
- 한국전기안전공사 — 전기안전 관제시스템 보안 모듈 개발
  - 발주기관: 한국전기안전공사
  - 사업명: 전기안전 관제시스템 보안 모듈 개발
  - 사업금액: 222,180,200원 (메타데이터)
  - 사업예산(문서 표기): 220,000천원(VAT 포함)
  - 사업기간: 계약 후 180일
  - 주요내용: KCMVP 암호모듈 적용, MQTT Broker 구성·통신(TLS) 및 데이터 연동규격 적용, 제조사 대상 컨설팅·교육, 지도형 대시보드 등 기능 개발
  [근거: 한국전기안전공사_전기안전 관제시스템 보안 모듈 개발 용역.hwp]

- 국방과학연구소 — 기록관리시스

In [18]:
recheck_v3 = [
    ("b1", "철도 시설을 가상 공간에 구현하기 위한 디지털 전환 전략을 수립하는 공공기관 사업은 무엇인가요?"),
    ("b3", "병원 전산 장애나 재해에 대비해 진료정보를 복구할 체계를 마련하는 적십자 관련 용역은 무엇인가요?"),
    ("b4", "학생들의 교과 밖 활동을 관리할 온라인 환경을 새로 만드는 을지대 사업은 무엇인가요?"),
    ("b10", "코레일이 승차권 예매·발매 플랫폼의 개편 방향을 세우기 위해 진행한 계획수립 용역은 무엇인가요?"),
    ("b20", "한국원자력연구원 선량평가시스템 사업이 왜 추진되는지 목적을 알려줘"),
    ("b22", "학교 발주 사업 중에서 예산이 가장 큰 곳은?"),
    ("b23", "봉화군 재난통합관리시스템과 충북연구원 재난안전데이터 사업의 공통점과 차이점은?"),
]

for label, q in recheck_v3:
    print(f"=== [{label}] {q} ===")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

=== [b1] 철도 시설을 가상 공간에 구현하기 위한 디지털 전환 전략을 수립하는 공공기관 사업은 무엇인가요? ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

사업명: 철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역  
발주기관: 국가철도공단

[근거: 국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp)]

=== [b3] 병원 전산 장애나 재해에 대비해 진료정보를 복구할 체계를 마련하는 적십자 관련 용역은 무엇인가요? ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

적십자병원 병원정보 재해복구시스템 구축 용역입니다.  
[근거: 대한적십자사 의료원_적십자병원 병원정보 재해복구시스템 구축 용역 .hwp]

=== [b4] 학생들의 교과 밖 활동을 관리할 온라인 환경을 새로 만드는 을지대 사업은 무엇인가요? ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

을지대학교 비교과시스템 개발 사업입니다. 주요 내용: 비교과 관리 시스템(비교과 운영현황·대시보드 등) 및 교육과정인증 관리 시스템을 구축하고 기존 학사정보시스템 및 통합행정시스템과 연동·통합관리되도록 재개발하는 사업입니다. 소스코드에 시큐어 코딩 적용, 기존 서버·프레임워크(Oracle, JEUS, Nexacro 등) 활용도 명시되어 있습니다.  
[근거: 을지대학교_을지대학교 비교과시스템 개발.hwp]

=== [b10] 코레일이 승차권 예매·발매 플랫폼의 개편 방향을 세우기 위해 진행한 계획수립 용역은 무엇인가요? ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

예약발매시스템 개량 ISMP 용역 — 예약발매시스템 개량을 위한 업무 종합계획(ISMP) 수립 용역입니다.  
[근거: 한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp]

=== [b20] 한국원자력연구원 선량평가시스템 사업이 왜 추진되는지 목적을 알려줘 ===
- 규제요건 준수: 원자력안전위원회 고시(방사선방호 기준)에서 요구하는 제한구역 경계 연간선량 준수 여부 확인 체계 구축 필요, ICRP60 기반 평가장기 개선 필요, 액체유출물로 인한 주민피폭 선량평가 수행 필요.  
- 업무생산성 향상: UI 개선으로 신속한 의사결정 환경 구축 및 출력 기능 고도화를 통해 생산성 향상.  
- 시스템 신뢰성 제고: 선량평가 시스템 고도화를 통해 데이터 신뢰성 확보 및 정상운전 시 규제 수요에 대응(프로젝트명: K-RADAC 고도화).  

[근거: 한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp]

=== [b22] 학교 발주 사업 중에서 예산이 가장 큰 곳은? ===
고려대학교 — 11,270,000,000원 (V.A.T 포함)

[근거: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf]

=== [b23] 봉화군 재난통합관리시스템과 충북연구원 재난안전데이터 사업의 공통점과 차이점은? ===
비교 항목별 정리 — 공통점 / 차이점

1) 사업명·목적  
- 봉화군: 재난통합관리시스템 고도화 사업 — GIS 기반 통합수집·관리, 통합상황전파·경보체계 고도화.  
- 충북연구원: GIS/통계 기반 재난안전데이터 분석·관리 시스템 구축 용역 — 공간·통계 기반 재난안전 데이터 통합·분석 및 정책지원.  
- 결론(공통): 둘 다 GIS 기반으로 재난정보의 통합·관리와 분석을 목표로 함. (차이점: 봉화군은 경보·전파(운영·방송·장비) 고도화에 중점, 충북연구원은 데이터 표준화·분석모델·정책지원·데이터품질 관리에 중점)

2) 발주기관·사업금액·계약방법·기간  
- 봉화군: 발주기관 경상북도 봉화군, 사업금액 900,000,000원(부가

In [19]:
sample_meta = next(c.metadata for c in child_chunks if c.doc_id == '한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp')
print(sample_meta)

{'발주_기관': '한국사회보장정보원', '사업_금액': 44000000.0, 'budget_unknown': False, '입찰_참여_마감일': '2025-02-27 14:00:00', '입찰참여마감일_결측': False, '파일형식': 'hwp', 'doc_type': 'plain_text', 'source': 'raw_parsed'}


In [20]:
b15_docs = ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp',
            '한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp',
            '인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp',
            '대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp',
            '광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp',
            '한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp',
            '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp',
            '고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp',
            '남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp',
            '부산관광공사_경영정보시스템 기능개선.hwp',
            '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp',
            '한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp']

for doc_id in b15_docs:
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    found = None
    for c in doc_c:
        m = re.search(r'(계약일|착수일)[^.]{0,10}로부터\s*(\d+)\s*(일|개월)', c.text)
        if m:
            found = m.group(0)
            break
    print(f"{doc_id[:40]}... -> {found}")

한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.h... -> 계약일로부터 3개월
한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp... -> 계약일로부터 90일
인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp... -> 계약일로부터 3개월
대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE... -> 계약일로부터 2개월
광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업... -> None
한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개... -> None
조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeH... -> None
고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp... -> 착수일로부터 90일
남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.h... -> 계약일로부터 3개월
부산관광공사_경영정보시스템 기능개선.hwp... -> 계약일로부터 90일
축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp... -> 계약일로부터 90일
한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.... -> 계약일로부터 90일


In [21]:
missed_docs = ['광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp',
               '한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp',
               '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp']

for doc_id in missed_docs:
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    for c in doc_c:
        if '개월' in c.text or ('일' in c.text and ('사업기간' in c.text or '용역기간' in c.text or '과업기간' in c.text)):
            idx = c.text.find('기간')
            print(f"{doc_id[:30]}...")
            print(c.text[max(0,idx-30):idx+60])
            print()
            break

광주과학기술원_실시간통합연구비관리시스템(RCMS)  연...
관리시스템(RCMS) 연계모듈 변경 사업
  나. 사업기간 : 계약체결일로부터 3개월 이내
      ※ 제안요청서 내 추진일정표 참조 
  다. 사업비  :  

한국지식재산보호원_IP-NAVI  해외지식재산센터 사업...
가신청 서류42[별첨 3]소프트웨어 개발사업 적정 사업기간 산정서54[별첨 4]소프트웨어사업 영향평가 검토결과서55[별첨 5]소프트웨어 과업변경요청서57[별첨 6

조선대학교_(재공고)2024 조선대학교 SW중심대학 사...
학교 SW중심대학 사업관리시스템(WeHub) 구축
사업기간 : 계약체결일로부터 90일(약 3개월)
하자유지보수기간: 사업을 종료한 날부터 1년
설계금액 : 금 1



In [22]:
def extract_period_days(doc_id, child_chunks):
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    for c in doc_c:
        m = re.search(r'(계약체결일|계약일|착수일)[^.]{0,10}로부터\s*(\d+)\s*(일|개월)', c.text)
        if m:
            num = int(m.group(2))
            unit = m.group(3)
            days = num * 30 if unit == '개월' else num
            return days, m.group(0)
    return None, None

days, matched = extract_period_days('한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp', child_chunks)
print(f"한국지식재산보호원: {days}일, 매칭: {matched}")

days2, matched2 = extract_period_days('광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp', child_chunks)
print(f"광주과학기술원: {days2}일, 매칭: {matched2}")

한국지식재산보호원: None일, 매칭: None
광주과학기술원: 90일, 매칭: 계약체결일로부터 3개월


In [24]:
doc_c = [c for c in child_chunks if c.doc_id == '한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp']
for i, c in enumerate(doc_c):
    if '개월' in c.text or ('기간' in c.text and '일' in c.text):
        print(f"[{i}] {c.text[:300]}")
        print("")

[0] [표]
제안요청서

[표]
사 업 명 | IP-NAVI해외지식재산센터 사업관리 시스템 기능개선
주관기관

2024. 6.

[표]
사업책임자 | 해외지재권종합지원실 | 실장 전충규 | TEL:02)6250-0865 | FAX:02)6250-0870
사업실무자 | 해외지재권종합지원실 | 전임 노승원 | TEL:02)6250-0868

[표]
목차

[표]
1.사업개요12.추진목적13.사업 추진계획14.제안요청 내용35.제안서 작성요령46.안내 사항5[참고]제안요청사항9별표1. 누출금지 대상정보별표2. 보안위반 처리기준별표3. 사업자 

[5] ㅇ 제안서의 각 페이지는 쉽게 참조할 수 있도록 페이지 하단 중앙에 일련번호를 붙이되, 각 장별로 구분하여 번호를 부여함

 ㅇ 제안서는 한글 작성이 원칙이며, 사용된 영문약어에 대해서는 약어 설명을 함께 제공해야 함

 ㅇ 제안서의 내용을 객관적으로 입증할 수 있는 관련 자료는 제안서의 별첨으로 제출하여야 함

 ㅇ 제안서의 내용은 명확한 용어를 사용하여 표현하여야 함. 예를 들어, ‘사용가능하다’, ‘할 수 있다’, ‘고려하고 있다’ 등과 같이 모호한 표현은 평가 시 불가능한 것으로 간주하게 되며, 계량화가 가능한 것은 계량화하

[6] - 제안서 본문 내용은 300페이지(150장) 이내로 작성

    - 제안 설명 시 홍보용 동영상 활용 금지

 ㅇ 기타 공고서 및 조달청 제안평가 관련 규정에 따름

6. 안내 사항

 가. 입찰방식

  ㅇ 기본 방침

    - 객관적이고 공정한 기준과 절차를 적용하여 경쟁에 의한 우수 사업자 선정

  ㅇ 입찰 참가 자격

    - 「국가를 당사자로 하는 계약에 관한 법률 시행령」제12조 규정(경쟁 입찰의 참가자격)에 의한 입찰참가 자격요건을 갖춘 업체
     * 제안 업체는 국가종합전자조달시스템 입찰참가자격등록규정(조

[16] [표]
요구사항 고유번호 | FUN-003
요구사항 명칭 | 지원사업 신청자 접속 화면 구축
요구사항 분류 | 기능 요구사항 | 응낙수준 

In [25]:
def extract_period_days_v2(doc_id, child_chunks):
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    for c in doc_c:
        m = re.search(r'(계약체결일|계약일|착수일)[^.]{0,10}(로부터|~)\s*(\d+)\s*(일|개월)', c.text)
        if m:
            num = int(m.group(3))
            unit = m.group(4)
            days = num * 30 if unit == '개월' else num
            return days, m.group(0)
    return None, None

days, matched = extract_period_days_v2('한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp', child_chunks)
print(f"한국지식재산보호원: {days}일, 매칭: {matched}")

한국지식재산보호원: 90일, 매칭: 계약체결일 ~ 3개월


In [26]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

new_functions = '''

def is_short_period_question(question):
    """'N개월/N일 이내로 짧은 사업' 같은 질문 감지"""
    return bool(re.search(r'(\\d+)\\s*(개월|일)\\s*이내', question)) or '짧은 사업' in question


def extract_period_threshold_days(question):
    """질문에서 기준 기간(일수)을 추출. 명시 안 되어 있으면 기본값 90일(3개월)"""
    m = re.search(r'(\\d+)\\s*(개월|일)\\s*이내', question)
    if m:
        num = int(m.group(1))
        unit = m.group(2)
        return num * 30 if unit == '개월' else num
    return 90


def extract_period_days(doc_id, child_chunks):
    """문서 본문에서 '계약일/착수일로부터(또는 ~) N일/N개월' 표현을 찾아 일수로 변환"""
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    for c in doc_c:
        m = re.search(r'(계약체결일|계약일|착수일)[^.]{0,10}(로부터|~)\\s*(\\d+)\\s*(일|개월)', c.text)
        if m:
            num = int(m.group(3))
            unit = m.group(4)
            return num * 30 if unit == '개월' else num
    return None

'''

marker = "def ask_rfp_v9("
idx = content.find(marker)
content = content[:idx] + new_functions.strip() + "\n\n\n" + content[idx:]

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("함수 추가 완료")

함수 추가 완료


In [27]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """    # 랭킹/집계형 질문(예: 예산 차이가 가장 작은 사업 찾기) 우선 처리
    if is_closest_budget_question(question):
        result = parse_closest_budget_query(question, all_filenames_with_biz, child_chunks, extract_doc_hints_multi)
        if result:
            fname, amt, diff = result
            return f"'{fname}' 사업입니다. 예산은 {amt:,.0f}원이며, 차이는 {diff:,.0f}원입니다.\\n\\n[근거: {fname}]"
"""

new_code = """    # 랭킹/집계형 질문(예: 예산 차이가 가장 작은 사업 찾기) 우선 처리
    if is_closest_budget_question(question):
        result = parse_closest_budget_query(question, all_filenames_with_biz, child_chunks, extract_doc_hints_multi)
        if result:
            fname, amt, diff = result
            return f"'{fname}' 사업입니다. 예산은 {amt:,.0f}원이며, 차이는 {diff:,.0f}원입니다.\\n\\n[근거: {fname}]"

    # 기간이 짧은 사업을 찾는 질문 우선 처리 (문서 본문에서 사업기간을 직접 파싱)
    if is_short_period_question(question):
        threshold = extract_period_threshold_days(question)
        matched_docs = []
        for fname, biz in all_filenames_with_biz:
            days = extract_period_days(fname, child_chunks)
            if days is not None and days <= threshold:
                matched_docs.append((fname, days))
        if matched_docs:
            lines = [f"- {fname} ({days}일)" for fname, days in matched_docs]
            doc_list_str = "\\n".join(lines)
            fname_list_str = ", ".join(fname for fname, _ in matched_docs)
            return f"다음 사업들이 {threshold}일 이내로 진행됩니다:\\n{doc_list_str}\\n\\n[근거: {fname_list_str}]"
"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [28]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(answer_generation)

q_b15 = "3개월 이내로 끝나는 짧은 사업 있어?"
answer = ask_rfp_v9(q_b15, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

다음 사업들이 90일 이내로 진행됩니다:
- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (90일)
- 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp (14일)
- 재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp (10일)
- (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp (10일)
- 한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp (10일)
- 고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp (90일)
- 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp (75일)
- 한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp (90일)
- 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp (10일)
- 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp (90일)
- 부산관광공사_경영정보시스템 기능개선.hwp (90일)
- 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp (80일)
- 대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp (60일)
- 한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp (10일)
- 서영대학교 산학협력단_전문대학 혁신지원사업 서영대학교 차세대 교육.hwp (14일)
- 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp (10일)
- 한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp (10일)
- 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp (14일)
- 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp (15일)
- 케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp (0일)
- 파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp

In [29]:
days, matched = None, None
for c in [c for c in child_chunks if c.doc_id == '케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp']:
    m = re.search(r'(계약체결일|계약일|착수일)[^.]{0,10}(로부터|~)\s*(\d+)\s*(일|개월)', c.text)
    if m:
        print(m.group(0))
        break

계약일로부터  00일


In [30]:
def extract_period_days_v3(doc_id, child_chunks):
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    for c in doc_c:
        # "사업기간", "용역기간", "과업기간" 근처(50자 이내)에 있는 경우만 인정
        m = re.search(r'(사업|용역|과업)\s*기간[^.]{0,30}(계약체결일|계약일|착수일)[^.]{0,10}(로부터|~)\s*(\d+)\s*(일|개월)', c.text)
        if m:
            num_str = m.group(4)
            if num_str == '00' or int(num_str) == 0:
                continue
            num = int(num_str)
            unit = m.group(5)
            return num * 30 if unit == '개월' else num
    return None

# 재검증: 정답 12개 + 오탐이었던 케빈랩
test_docs = b15_docs + ['케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp']
for doc_id in test_docs:
    days = extract_period_days_v3(doc_id, child_chunks)
    print(f"{doc_id[:40]}... -> {days}")

한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.h... -> 90
한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp... -> 90
인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp... -> 90
대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE... -> 60
광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업... -> 90
한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개... -> 90
조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeH... -> 90
고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp... -> 90
남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.h... -> 90
부산관광공사_경영정보시스템 기능개선.hwp... -> 90
축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp... -> 90
한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.... -> 90
케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .... -> None


In [31]:
b15_docs = ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp',
            '한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp',
            '인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp',
            '대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp',
            '광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp',
            '한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp',
            '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp',
            '고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp',
            '남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp',
            '부산관광공사_경영정보시스템 기능개선.hwp',
            '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp',
            '한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp']

def extract_period_days_v3(doc_id, child_chunks):
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    for c in doc_c:
        m = re.search(r'(사업|용역|과업)\s*기간[^.]{0,30}(계약체결일|계약일|착수일)[^.]{0,10}(로부터|~)\s*(\d+)\s*(일|개월)', c.text)
        if m:
            num_str = m.group(4)
            if int(num_str) == 0:
                continue
            num = int(num_str)
            unit = m.group(5)
            return num * 30 if unit == '개월' else num
    return None

print("정답 12개")
for doc_id in b15_docs:
    days = extract_period_days_v3(doc_id, child_chunks)
    print(f"{doc_id[:40]}... -> {days}")

print("\n오탐이었던 케빈랩")
print(extract_period_days_v3('케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp', child_chunks))

정답 12개
한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.h... -> 90
한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp... -> 90
인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp... -> 90
대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE... -> 60
광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업... -> 90
한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개... -> 90
조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeH... -> 90
고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp... -> 90
남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.h... -> 90
부산관광공사_경영정보시스템 기능개선.hwp... -> 90
축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp... -> 90
한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.... -> 90

오탐이었던 케빈랩
None


In [32]:
with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'r', encoding='utf-8') as f:
    content = f.read()

old_code = """def extract_period_days(doc_id, child_chunks):
    \"\"\"문서 본문에서 '계약일/착수일로부터(또는 ~) N일/N개월' 표현을 찾아 일수로 변환\"\"\"
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    for c in doc_c:
        m = re.search(r'(계약체결일|계약일|착수일)[^.]{0,10}(로부터|~)\\s*(\\d+)\\s*(일|개월)', c.text)
        if m:
            num = int(m.group(3))
            unit = m.group(4)
            return num * 30 if unit == '개월' else num
    return None"""

new_code = """def extract_period_days(doc_id, child_chunks):
    \"\"\"문서 본문에서 '사업/용역/과업기간 ... 계약일/착수일로부터(또는 ~) N일/N개월' 표현을
    찾아 일수로 변환. '00일'처럼 채워지지 않은 서식은 제외.\"\"\"
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    for c in doc_c:
        m = re.search(r'(사업|용역|과업)\\s*기간[^.]{0,30}(계약체결일|계약일|착수일)[^.]{0,10}(로부터|~)\\s*(\\d+)\\s*(일|개월)', c.text)
        if m:
            num_str = m.group(4)
            if int(num_str) == 0:
                continue
            num = int(num_str)
            unit = m.group(5)
            return num * 30 if unit == '개월' else num
    return None"""

count = content.count(old_code)
print(f"찾은 개수: {count}")

content = content.replace(old_code, new_code)

with open('/content/drive/MyDrive/중급 프로젝트/answer_generation.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("수정 완료")

찾은 개수: 1
수정 완료


In [33]:
import importlib
import answer_generation
from answer_generation import ask_rfp_v9

importlib.reload(answer_generation)

q_b15 = "3개월 이내로 끝나는 짧은 사업 있어?"
answer = ask_rfp_v9(q_b15, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
print(answer)

다음 사업들이 90일 이내로 진행됩니다:
- 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp (90일)
- 고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp (90일)
- 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp (75일)
- 한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp (90일)
- 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp (90일)
- 부산관광공사_경영정보시스템 기능개선.hwp (90일)
- 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp (80일)
- 대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp (60일)
- 파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp (60일)
- 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp (90일)
- 남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp (90일)
- 축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp (90일)
- 한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp (90일)
- 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp (90일)
- 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp (90일)

[근거: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp, 고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp, 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp, 한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp, 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp, 부산관광공사_경영정보시스템 기능개선.hwp, 전북특별자치도 정읍시_

In [34]:
extra_docs = {'파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp'}
expected_b15 = set(b15_docs)
answer_b15 = {'한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp',
              '고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp',
              '전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp',
              '한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp',
              '인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp',
              '부산관광공사_경영정보시스템 기능개선.hwp',
              '전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp',
              '대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp',
              '파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp',
              '광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp',
              '남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp',
              '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp',
              '한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp',
              '한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp',
              '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp'}

print("빠진 것:", expected_b15 - answer_b15)
print("초과된 것:", answer_b15 - expected_b15)

빠진 것: set()
초과된 것: {'전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp', '전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp', '파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp'}


In [35]:
set_items_b15 = golden_v3[golden_v3['id'] == 'supplemental-set-b15']
row = set_items_b15.iloc[0]
print("정답 문서 목록:", row['expected_doc_id'])
print()
print("required_fact_groups:", row['required_fact_groups'])

정답 문서 목록: ['한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp', '한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp', '인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp', '대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp', '광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp', '한국지식재산보호원_IP-NAVI  해외지식재산센터 사업관리 시스템 기능개.hwp', '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp', '고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp', '남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp', '부산관광공사_경영정보시스템 기능개선.hwp', '축산물품질평가원_꿀 품질평가 전산시스템 기능개선 사업.hwp', '한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp']

required_fact_groups: [['한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화'], ['라오스 보건의료정보화 협력을 위한 사전타당성 조사'], ['인천일자리플랫폼 정보시스템 구축 ISP 수립용역'], ['대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전산시스템 구축'], ['실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업'], ['IP-NAVI 해외지식재산센터 사업관리 시스템 기능개선'], ['(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구축'], ['관산근린공원 다목적구장 홈페이지 및 회원 통합운영 관리 시스템 구축[협상에 의한 계약]'], ['[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사행정 암호화) 개발 용역 입찰'], ['경영정보시스템 기능개선'], ['꿀 품질평가 전산시스템 기능개선 사업'], ['R

In [37]:
for doc_id in ['전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp',
               '전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp',
               '파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp']:
    doc_c = [c for c in child_chunks if c.doc_id == doc_id]
    for c in doc_c:
        m = re.search(r'(사업|용역|과업)\s*기간[^.]{0,60}', c.text)
        if m and ('일' in m.group(0) or '개월' in m.group(0)):
            print(f"{doc_id[:30]}...")
            print(m.group(0))
            print()
            break

전북대학교_JST 공유대학(원) xAPI기반 LRS시스...
사업기간: 계약체결일로부터 75일까지
□ 사업예산: 금116,000,000원(금일억일천육백만원, 부가세 포함)
□ 

전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시...
과업기간 : 착수일로부터 80일 이내
 3

파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업...
사업기간 : 착수일로부터 60일
    라



In [38]:
visual_test_questions = golden_v3[golden_v3['source_lane'] == 'visual']

for _, row in visual_test_questions.iterrows():
    print(f"[{row['id']}] {row['query']}")

[visual-hwp-table-001] 수문자료정보관리시스템(HDIMS) 재구축 용역(3단계)의 기술성 평가 구성표에서 정량적 평가와 정성적 평가의 소계 및 평가 방식은 각각 무엇인가?
[visual-hwp-table-002] 실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업에서 입찰공고일 현재 4대 보험 납입 기준 보유인력이 정확히 20명인 업체는 보유인력 항목에서 몇 점을 받으며, 이 항목의 배점한도는 몇 점인가?
[visual-hwp-table-003] 국방과학연구소 기록관리시스템 사업의 요구사항 목록 전체(렌더 9~10쪽)를 기준으로, 요구사항 수가 가장 많은 구분과 가장 적은 구분(동률 포함)은 무엇이며 각각 몇 건인가? 총 요구사항 수도 함께 답하라.
[visual-hwp-figure-001] 국방과학연구소 기록관리시스템의 대상 시스템 현황 그림에서 웹서버 제품·버전, DBMS 제품·버전, Active/Standby 운영서버의 운영체제를 각각 답하라.
[visual-hwp-figure-002] 네팔 수자원관리 Pilot 시스템의 운영체계 그림에서 GIDC의 Main server와 DWRI의 Secondary server 사이 백업 주기와 DWRI 서버 유지보수 주기는 각각 어떻게 표시되어 있는가?
[visual-pdf-table-001] 서울시립대학교의 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역에서, 유사사업 최대 실적의 규모비율이 정확히 70%라면 수행실적 금액의 환산점수는 몇 점인가?
[visual-pdf-table-002] 기초과학연구원의 중이온가속기용 극저온시스템 운전 용역에서, 연구원 승인이 필요하면서 ‘발생한 경우에만’ 제출하는 문서는 무엇이며 제출·회람·저장 조건은 무엇인가?
[visual-pdf-figure-001] 서울시 지도정보 플랫폼 시스템 개념도에서 내부 지도정보 플랫폼으로 들어오는 데이터 종류 3가지와 시민용 스마트서울맵이 제공하는 대표 지도 서비스 2가지를 각각 말해라.
[visual-pdf-table-

In [39]:
visual_test = [
    "수문자료정보관리시스템(HDIMS) 재구축 용역(3단계)의 기술성 평가 구성표에서 정량적 평가와 정성적 평가의 소계 및 평가 방식은 각각 무엇인가?",
    "실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업에서 입찰공고일 현재 4대 보험 납입 기준 보유인력이 정확히 20명인 업체는 보유인력 항목에서 몇 점을 받으며, 이 항목의 배점한도는 몇 점인가?",
    "국방과학연구소 기록관리시스템의 대상 시스템 현황 그림에서 웹서버 제품·버전, DBMS 제품·버전, Active/Standby 운영서버의 운영체제를 각각 답하라.",
    "네팔 수자원관리 Pilot 시스템의 운영체계 그림에서 GIDC의 Main server와 DWRI의 Secondary server 사이 백업 주기와 DWRI 서버 유지보수 주기는 각각 어떻게 표시되어 있는가?",
]

for q in visual_test:
    print(f"{q}")
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    print(answer)
    print()

수문자료정보관리시스템(HDIMS) 재구축 용역(3단계)의 기술성 평가 구성표에서 정량적 평가와 정성적 평가의 소계 및 평가 방식은 각각 무엇인가?
정량적 평가: 소계 10점, 평가방식 = 절대평가 (주요항목 예: 유사분야 수행실적 5점, 경영상태·신용평가 5점 등)  
정성적 평가: 소계 90점, 평가방식 = 상대평가(또는 절대평가 병행 가능) — 세부항목은 전략·기술·성능·프로젝트관리·프로젝트지원 등으로 구성되며 각 항목은 최고 5등급(평가위원회 판단에 따라 절대평가 또는 상대평가 실시).  
[근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp)]

실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업에서 입찰공고일 현재 4대 보험 납입 기준 보유인력이 정확히 20명인 업체는 보유인력 항목에서 몇 점을 받으며, 이 항목의 배점한도는 몇 점인가?
보유인력 항목 점수: 4.625점  
배점한도: 5점

[근거: 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp]

국방과학연구소 기록관리시스템의 대상 시스템 현황 그림에서 웹서버 제품·버전, DBMS 제품·버전, Active/Standby 운영서버의 운영체제를 각각 답하라.
확인되지 않습니다.  
[근거: 국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp]

네팔 수자원관리 Pilot 시스템의 운영체계 그림에서 GIDC의 Main server와 DWRI의 Secondary server 사이 백업 주기와 DWRI 서버 유지보수 주기는 각각 어떻게 표시되어 있는가?
확인되지 않습니다.

[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]



In [40]:
final_am_check_40 = []
for item in core40:
    task_type = item['task_type']
    question = item['question']
    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    answer = ask_rfp_v9(combined_q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_am_check_40.append({'case_id': item['case_id'], 'task_type': task_type, 'answer': answer})
    print(f"[{item['case_id']}][{task_type}] {question}")
    print(answer)
    print()

[dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산은 243,000,000원이며 부가가치세(VAT) 포함 표기입니다.  
[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)  
예산: 70,000,000원(금칠천만원, VAT 포함)

[근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp]

[dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차(1차: 계약일로부터 4개월, 2차: 1차 완료일로부터 2개월)로 나뉩니다.
- 평가비중은 기술평가 90%, 가격평가 10%입니다.

[근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp]

[dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
사업기간: 계약일로부터 90일(= 계약일로부터 3개월)  
시범 도입 규모: 1단계 시범도입 3개 기관(서울 2개소, 울산 1개소)

[근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp]

[dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출 방법: 입찰서와 제안서는 전자적으로 나라장터(e-발주시스템)를 통해 제출해야 함. 제안서는 입찰서 제출기한과 동일함. ([서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp] 근거 p. 관련 섹션)
- 파일 형식: 나라장터를 통해 제출하는 제안서류 일체

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
이 기능은 제가 수행할 수 없습니다.

[근거: 한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp, 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp, 한국생산기술연구원_2세대 전자조달시스템  기반구축사업.hwp, 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp, 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp, 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시스템 구.hwp]

[dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다. [근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp)]

[dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
이 문서만으로 귀사가 자격요건을 충족하는지 판정할 수 없습니다.

대신 판단을 위해 귀사가 스스로 확인해야 할 필수 조건만 간단히 적어드립니다.
- 지방자치단체 계약법령상 부정당업자 해당사항이 없는지 확인.  
- 입찰공고일 전일부터 계약체결일까지 주된 영업소 소재지가 부산광역시인지 확인.  
- 나라장터(G2B)에 입찰서 제출마감일 전까지 소프트웨어사업자(업종코드 1468)로 입찰참가자격 등록이 완료되어 있는지 확인.  
- 소프트웨어산업 진흥법 등 관련 규정에 따라 대기업·중견기업(및 상호출자제한기업집단 소속회사)이 아닌 중소 소프트웨어사업자인지 확인.  
- 정보시스템개발서비스(세부품명번호 8111159901) 관련 ‘직접생산확인증명서’를 입찰마감 전까지 발급받아 유효기간 내에 있는지 확인.  
- 공동수급(공동이행방식), 하도급이 허용되지 않으므로 단독 참여 가능 여부 확인.

위 항목들에 대한 귀사 내부 증빙(사업자등록상 주된 영업소 주소, G2B 등록 확인서, 직접생산확인증명서 등)을 대조하면 자격 충족 여부를 판단할 수 있습니다.  
문서에 명시된 다른 세부조건(예: 제출서류 목록의 개별 요건 등)은 원문 전체 확인이 필요합니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[dev-unknown-005][unknown] 서민금융 채팅상담 사업에 경쟁사가 몇 곳 참여할 예정이야?
확인되지 않습니다.

[근거: 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 구축.hwp]

[dev-unknown-006][unknown]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
답변할 수 없다

[근거: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp, 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp, 부산관광공사_경영정보시스템 기능개선.hwp, 한국로봇산업진흥원_한국로봇산업진흥원 사업관리시스템 온라인평가 .hwp, 국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp]

[dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
문서만으로 낙찰확률을 퍼센트로 계산할 수 없습니다. 이 판단은 귀사(또는 경쟁사)의 역량·가격·제출서류·참여자격 충족 여부 등 외부정보와 문서 외 변수를 대조·분석해야 하며, 제공된 문서만으로는 계산 근거가 없습니다.

필요한 판단 항목(간단히):
- 귀사의 제출 가격(또는 가격 정책)과 예상 경쟁사 가격 비교
- 귀사의 제안서(기술·경험·인력)과 각 사업 요구사항(기술요구, 파견인력, 연동·공급 항목) 일치 여부
- 참가자격(지역제한 등) 충족 여부
- 과업 범위(예: BIFF는 웹·앱·행사지원 시스템 다수, 파견인력 요구 등) 대비 수행 능력
- 계약방법·입찰방식(예: BIFF는 제한경쟁·협상에 의한 계약, 서민금융은 제한경쟁·긴급공고)로 인한 경쟁 구도 차이
- 제출서류의 완전성·평가기준(배점) 및 과거 유사사업 수행실적
- 일정(제안서 제출·발표 일정)과 준비 가능성

원하시면 위 항목들에 대해 귀사의 구체적 정보(가격, 보유인력, 과거 수행실적, 준비상태 등)를 주시면, 문서 근거 항목과 대조해 "평가에 필요한 조건 목록"을 바탕으로 비교 가능한 요소를 정리해 드

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
문서에 없는 정보는 임의로 가정해서 확정할 수 없습니다

[근거: 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp, 수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp, 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp, 한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp, 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp, 한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp, 한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp, 한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp]

[dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
실시간 진행률 및 계약 지연 가능성에 대한 정보는 제공된 문서에서 확인할 수 없습니다. 이 기능은 제가 수행할 수 없습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



In [42]:
def normalize_text(t):
    return t.replace(',', '').replace(' ', '')

def normalize_dates(text):
    text = re.sub(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', r'\1.\2.\3', text)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', lambda m: f"{m.group(1)}.{int(m.group(2)):02d}.{int(m.group(3)):02d}", text)
    return text

def _text_included(fact_text, answer_text):
    fact_text = normalize_dates(fact_text)
    answer_text_n = normalize_dates(answer_text)
    answer_norm = normalize_text(answer_text_n)

    numbers = re.findall(r'\d+(?:\.\d+)?', fact_text)
    numbers = [n for n in numbers if len(n) >= 2]

    if numbers:
        all_numbers_match = True
        for num in numbers:
            if num in answer_norm:
                continue
            if len(num) == 4 and num.startswith('20'):
                if num[2:] in answer_norm:
                    continue
            num_no_zero = re.sub(r'^0+', '', num)
            if num_no_zero and num_no_zero in answer_norm:
                continue
            all_numbers_match = False
            break

        if not all_numbers_match:
            return False

        raw_words = re.split(r'[\s,·:()]+', fact_text)
        stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
        core_words = []
        for w in raw_words:
            w = w.rstrip('.,')
            if len(w) < 2:
                continue
            if re.match(r'^\d', w):
                continue
            for suf in stopwords_suffix:
                if w.endswith(suf) and len(w) > len(suf):
                    w = w[:-len(suf)]
                    break
            w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
            if len(w) >= 2:
                core_words.append(w)

        if not core_words:
            return True

        match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
        return match_count / max(len(core_words), 1) >= 0.2

    raw_words = re.split(r'[\s,·:()]+', fact_text)
    stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
    core_words = []
    for w in raw_words:
        w = w.rstrip('.,')
        if len(w) < 2:
            continue
        if re.match(r'^\d', w):
            continue
        for suf in stopwords_suffix:
            if w.endswith(suf) and len(w) > len(suf):
                w = w[:-len(suf)]
                break
        w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
        if len(w) >= 2:
            core_words.append(w)

    if not core_words:
        return True

    match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
    return match_count / max(len(core_words), 1) >= 0.4

ABSTAIN_PHRASES = ['확인되지 않습니다', '답변할 수 없', '수행할 수 없', '확인할 수 없', '판단할 수 없',
                    '판정할 수 없', '판정해줄 수 없', '계산할 수 없', '제공할 수 없', '받아들일 수 없',
                    '확정할 수 없', '알려드릴 수 없', '불가능합니다', '제공된 문서 범위에서는',
                    '해드릴 수 없', '드릴 수 없']

def official_score_core40(item, answer_text):
    gold = item['gold']
    decision = gold.get('decision')

    if decision == 'abstain':
        is_abstained = any(p in answer_text for p in ABSTAIN_PHRASES)
        return 100 if is_abstained else 0

    key_points = gold.get('required_key_points', [])
    if not key_points:
        return None

    included = [_text_included(kp['text'], answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

In [43]:
for r in final_am_check_40:
    item = next(it for it in core40 if it['case_id'] == r['case_id'])
    r['score'] = official_score_core40(item, r['answer'])
    print(f"[{r['case_id']}][{r['task_type']}] 점수: {r['score']}")

valid_scores = [r['score'] for r in final_am_check_40 if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

[dev-single-001][single_doc] 점수: 100.0
[dev-single-002][single_doc] 점수: 100.0
[dev-single-003][single_doc] 점수: 100.0
[dev-single-004][single_doc] 점수: 100.0
[dev-single-005][single_doc] 점수: 100.0
[dev-single-006][single_doc] 점수: 100.0
[dev-single-007][single_doc] 점수: 100.0
[dev-single-008][single_doc] 점수: 100.0
[dev-single-009][single_doc] 점수: 100.0
[dev-single-010][single_doc] 점수: 66.67
[dev-multi-001][multi_doc_compare] 점수: 100.0
[dev-multi-002][multi_doc_compare] 점수: 100.0
[dev-multi-003][multi_doc_compare] 점수: 100.0
[dev-multi-004][multi_doc_compare] 점수: 100.0
[dev-multi-005][multi_doc_compare] 점수: 100.0
[dev-multi-006][multi_doc_compare] 점수: 100.0
[dev-multi-007][multi_doc_compare] 점수: 100.0
[dev-multi-008][multi_doc_compare] 점수: 100.0
[dev-multi-009][multi_doc_compare] 점수: 100.0
[dev-multi-010][multi_doc_compare] 점수: 75.0
[dev-followup-001][follow_up] 점수: 100.0
[dev-followup-002][follow_up] 점수: 50.0
[dev-followup-003][follow_up] 점수: 100.0
[dev-followup-004][follow_up] 점수: 100.0
[d

In [44]:
final_am_check_56 = []
for item in rag56:
    question = item['question']
    answer = ask_rfp_v9(question, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_am_check_56.append({'case_id': item['case_id'], 'answer': answer})
    print(f"[{item['case_id']}] {question}")
    print(answer)
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c01] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원(부가세 포함)

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[supplemental-qa-c02] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024. 10. 31. 까지 완료해야 합니다.  
[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[supplemental-qa-c03] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
사업예산은 1,515,000천원(부가세 포함) — 즉 1,515,000,000원입니다.  
[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[supplemental-qa-c04] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
입찰방식: 제한경쟁입찰(국가를 당사자로 하는 계약에 관한 법률 제7조)  
낙찰(사업자선정)절차: 협상에 의한 계약(「국가를 당사자로 하는 계약에 관한 법률 시행령」 제43조 및 관련 계약예규)에 따라 사업자 선정.  
(평가방식: 기술평가 90% / 가격평가 10%)

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[supplemental-qa-c05] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월입니다.  
[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[supplemental-qa-c06] 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성한 예산은 얼마인가요? 세금 포함 여부도 알려주세요.
181,913,000원 (VAT 포함)  
[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[supplemental-qa-c07] 인천공항운영서비스가 회계·인사 등 경영업무를 통합할 차세

In [45]:
for r in final_am_check_56:
    item = next(it for it in rag56 if it['case_id'] == r['case_id'])
    matched, total = check_required_facts(r['answer'], item['gold'].get('required_fact_groups'))
    r['score'] = round(matched / total * 100, 2) if total else None
    print(f"[{r['case_id']}] 점수: {r['score']}")

valid_scores_56 = [r['score'] for r in final_am_check_56 if r['score'] is not None]
print(f"\nrag-56 전체 평균: {sum(valid_scores_56)/len(valid_scores_56):.2f}/100 ({len(valid_scores_56)}개)")

[supplemental-qa-c01] 점수: 100.0
[supplemental-qa-c02] 점수: 100.0
[supplemental-qa-c03] 점수: 100.0
[supplemental-qa-c04] 점수: 100.0
[supplemental-qa-c05] 점수: 100.0
[supplemental-qa-c06] 점수: 100.0
[supplemental-qa-c07] 점수: 100.0
[supplemental-qa-c08] 점수: 100.0
[supplemental-qa-c09] 점수: 100.0
[supplemental-qa-c10] 점수: 100.0
[supplemental-qa-c11] 점수: 100.0
[supplemental-qa-c12] 점수: 66.67
[supplemental-qa-c13] 점수: 100.0
[supplemental-qa-c14] 점수: 100.0
[supplemental-qa-c15] 점수: 66.67
[supplemental-qa-c16] 점수: 50.0
[supplemental-qa-c18] 점수: 100.0
[supplemental-qa-c19] 점수: 0.0
[supplemental-qa-c20] 점수: 50.0
[supplemental-qa-c23] 점수: 0.0
[supplemental-qa-c25] 점수: 50.0
[supplemental-qa-g01] 점수: 100.0
[supplemental-qa-g02] 점수: 100.0
[supplemental-qa-g03] 점수: 50.0
[supplemental-qa-g04] 점수: 100.0
[supplemental-qa-g05] 점수: 100.0
[supplemental-qa-g06] 점수: 100.0
[supplemental-qa-g07] 점수: 100.0
[supplemental-qa-g08] 점수: 80.0
[supplemental-qa-g11] 점수: 66.67
[supplemental-qa-g12] 점수: 75.0
[supplemental-qa-g

In [46]:
final_am_check_set13 = []
set_items_full = golden_v3[golden_v3['source_lane'] == 'set']

for _, row in set_items_full.iterrows():
    q = row['query']
    answer = ask_rfp_v9(q, client=client, index=index, child_chunks=child_chunks, all_filenames_with_biz=all_filenames_with_biz)
    final_am_check_set13.append({'id': row['id'], 'query': q, 'answer': answer, 'expected': row['expected_doc_id']})

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [47]:
for item in final_am_check_set13:
    print(f"[{item['id']}] {item['query']}")
    print(item['answer'])
    print()

[supplemental-set-b1] 철도 시설을 가상 공간에 구현하기 위한 디지털 전환 전략을 수립하는 공공기관 사업은 무엇인가요?
사업명: 철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역 — 발주기관: 국가철도공단.  
목적(요약): 철도 시설을 현실-가상 융합(디지털트윈)으로 구현하여 철도 인프라의 디지털 전환 전략·표준정보체계·플랫폼 구축 방안 및 중장기 로드맵(ISP) 수립을 위한 사업입니다.  
[근거: 국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp)]

[supplemental-set-b10] 코레일이 승차권 예매·발매 플랫폼의 개편 방향을 세우기 위해 진행한 계획수립 용역은 무엇인가요?
예약발매시스템 개량 ISMP 용역(정보시스템 마스터플랜(ISMP) 수립)

[근거: 한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp]

[supplemental-set-b12] 학교(대학교/대학/과학기술원)에서 발주한 시스템 사업 알려줘
다음 문서들이 학교(대학교/대학/과학기술원)에서 발주한 시스템 관련 사업입니다.

- 한영대학 — 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화, 사업예산 130,000,000원  
- 고려대학교 — 차세대 포털·학사 정보시스템 구축사업, 사업예산 11,270,000,000원  
- 서울시립대학교 — 학업성취도 다차원 종단분석 통합시스템 1차 고도화, 사업비 242,900,000원  
- 경희대학교 산학협력단 — 산학협력단 정보시스템(인포21) 운영 용역업체 선정, 사업예산 400,000,000원  
- 전북대학교(주관: 전북지역혁신플랫폼) — JST 공유대학(원) xAPI 기반 LRS 시스템 구축, 사업예산 116,000,000원  
- 광주과학기술원 — 학사시스템 기능개선 사업, 사업비 157,300,000원  
- 광주과학기술원 — 실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업, 사업비 54,450,000원  
- 을지대학교 — 비교

In [48]:
def evaluate_set_answer(expected_docs, answer_text):
    """답변 텍스트(근거 목록)에서 언급된 문서와 정답 문서를 비교해 Precision/Recall/F1 계산"""
    # 답변에서 언급된 문서 찾기 (정답 목록 안의 문서명이 답변에 포함되는지 체크)
    all_docs_in_answer = [doc for doc, _ in all_filenames_with_biz if doc in answer_text]

    expected_set = set(expected_docs)
    found_set = set(all_docs_in_answer)

    tp = len(expected_set & found_set)
    fp = len(found_set - expected_set)
    fn = len(expected_set - found_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, tp, fp, fn

results_set13 = []
for item in final_am_check_set13:
    p, r, f1, tp, fp, fn = evaluate_set_answer(item['expected'], item['answer'])
    results_set13.append({'id': item['id'], 'precision': p, 'recall': r, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn})
    print(f"[{item['id']}] P={p:.2f} R={r:.2f} F1={f1:.2f} (정답 {tp}개 찾음, 오탐 {fp}개, 누락 {fn}개)")

avg_p = sum(r['precision'] for r in results_set13) / len(results_set13)
avg_r = sum(r['recall'] for r in results_set13) / len(results_set13)
avg_f1 = sum(r['f1'] for r in results_set13) / len(results_set13)
print(f"\n평균 Precision: {avg_p:.3f}, 평균 Recall: {avg_r:.3f}, 평균 F1: {avg_f1:.3f}")

[supplemental-set-b1] P=1.00 R=1.00 F1=1.00 (정답 1개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b10] P=1.00 R=1.00 F1=1.00 (정답 1개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b12] P=1.00 R=1.00 F1=1.00 (정답 12개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b14] P=1.00 R=1.00 F1=1.00 (정답 5개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b15] P=0.80 R=1.00 F1=0.89 (정답 12개 찾음, 오탐 3개, 누락 0개)
[supplemental-set-b16] P=1.00 R=1.00 F1=1.00 (정답 2개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b20] P=1.00 R=1.00 F1=1.00 (정답 1개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b21] P=1.00 R=1.00 F1=1.00 (정답 1개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b22] P=1.00 R=1.00 F1=1.00 (정답 1개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b23] P=1.00 R=1.00 F1=1.00 (정답 2개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b24] P=1.00 R=1.00 F1=1.00 (정답 8개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b3] P=1.00 R=1.00 F1=1.00 (정답 1개 찾음, 오탐 0개, 누락 0개)
[supplemental-set-b4] P=1.00 R=1.00 F1=1.00 (정답 1개 찾음, 오탐 0개, 누락 0개)

평균 Precision: 0.985, 평균 Recall: 1.000, 평균 F1: 0.991
